# Verified Phase 2 lineage — Natural Sampling Phase 1

This notebook belongs to the corrected AOI-masked Phase 2 workflow. It must use only:

- `Models/Phase2_Harmonized_GEDIAnchored_NaturalP1` for Phase 2 checkpoints;
- `Results/Final_Article_Harmonized_GEDIAnchored_NaturalP1` for evaluation products;
- `Inference_Harmonized_GEDIAnchored_NaturalP1` for annual maps.

The former `Phase2_AOI_Masked`, `Final_Article_AOI_Masked`, and `Inference_AOI_Masked` products were generated from an incorrect Phase 1 parent lineage and must not be used. Run `Phase_2.ipynb` first, followed by `Inference.ipynb`, before regenerating downstream figures.


> **Active lineage (2026-08-09).** This notebook uses the harmonised GEDI-anchored Phase 2 checkpoints. Training sequences contain at least one valid GEDI observation, while dense image-only sequences are reserved for wall-to-wall inference. Execute the notebook from the first cell; outputs from the former AOI-only Phase 2 lineage are not reused.


# Natural Sampling publication pipeline

This notebook is the isolated Natural Sampling copy. The original official pipeline remains unchanged. Training is disabled because the selected Phase 1 and Phase 2 models are already archived under `Natural_Sampling/Models`. Run the notebook from the first cell to regenerate figures and tables under `Natural_Sampling/Results`.


> Execution order: regenerate the Natural Sampling annual Phase 2 CHMs with `Phase_2.ipynb` before running this benchmark notebook. The notebook intentionally stops if those Natural Sampling inference rasters are absent; it never falls back to the old official model outputs.


# CHM comparison on canonical GEDI TEST support

This notebook implements two complementary and pre-declared temporal protocols.

## Protocol A — primary multiannual benchmark

- Every available observation from the frozen canonical GEDI TEST split is retained.
- **Our Model is evaluated year by year:** a GEDI footprint acquired in year `Y` is sampled from the annual Phase-2 map for the same year `Y`.
- Static global products are sampled on the same multiannual TEST population. Their nominal dates are reported explicitly.
- Each product keeps its maximum valid raster support; paired comparisons use exact `shot_id` intersections.

This is the primary article analysis because it preserves the full independent TEST population and avoids choosing a favourable year.

## Protocol B — strict temporal sensitivity analysis

- Potapov/GFCH 2019 is compared with Our Model 2019 using only GEDI TEST observations from 2019.
- Lang 2020 is compared with Our Model 2020 using only GEDI TEST observations from 2020.
- Both members of a pair use exactly the same GEDI footprints.
- Meta/Tolan is not assigned an artificial year because the global product is based on imagery acquired at varying dates.

The strict-year results are a sensitivity analysis, not a replacement for Protocol A. No model or parameter is selected on TEST.

Spatial map panels remain in `Inference.ipynb`; this notebook creates only quantitative comparison figures and tables.


In [ ]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import shapely
from IPython.display import display
from matplotlib.patches import Patch
from rasterio.warp import transform
from rasterio.windows import Window, from_bounds
from shapely.geometry import Point

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
PRODUCT_ROOT = PROJECT / "CHM_Products_Comparison"
OUT_DIR = PROJECT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "CHM_Comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PHASE1_EXPECTED_SHA256 = {
    "Ifran": "072a8735973e3511564cf3ef7907f5b33105df08895c78917fb817de6198afb1",
    "Maamoura": "d83a5493715451a61c997a66a25f361080a56e7ceade60eb886f2ae76f6bd7f4",
    "Agadir": "f72f06e33455852cbc94cd00ecb81e192f68e7f67b89de007fef4a610fa25b2e",
}

def inference_manifest_path(map_path):
    return Path(map_path).parent.parent / "QA" / "inference_manifest.json"


RADIUS_M = 12.5
MIN_COVERAGE = 0.80
RECOMPUTE_AGGREGATION = False

COLORS = {
    "Our B4 Phase 2": "#78A6C8",
    "Lang 2020": "#DD8A6B",
    "GFCH 2019": "#A8D5E5",
    "Meta/Tolan 2023": "#DEA6B8",
    "Pauls 2020": "#E5D58A",
}
PRODUCT_LEGENDS = {
    "Lang 2020": "Lang et al. 2023 (L23; 10 m; map 2020)",
    "GFCH 2019": "Potapov et al. 2021 (P21; 30 m; map 2019)",
    "Meta/Tolan 2023": "Tolan et al. 2024 (T24; 10 m; variable dates)",
    "Pauls 2020": "Pauls et al. 2024 (Pa24; 10 m; map 2020)",
}

PRODUCT_NOMINAL_YEARS = {
    "Our B4 Phase 2": None,   # annual, exactly matched to each GEDI acquisition year
    "Lang 2020": 2020,
    "GFCH 2019": 2019,
    "Meta/Tolan 2023": None,  # varying imagery dates; 2023 is not a map snapshot
    "Pauls 2020": 2020,
}
OUR_PRODUCT = "Our B4 Phase 2"
ANNUAL_MAP_YEARS = tuple(range(2018, 2026))

SITES = {
    "Ifran": {
        "key": "Ifran_6",
        "ecosystem": "dense forest",
        "map_year": 2020,
        "eval_max": 45.0,
        "height_edges": [0, 5, 10, 15, 20, 25, 30, 35, 40, 45],
        "catalog": Path(r"C:\Users\Dell\Desktop\Publication_Clarck") / "Data/Dense/Ifran/Catalogs/final_catalog_C15_NATIVE",
        "ours": PROJECT / "Inference_Harmonized_GEDIAnchored_NaturalP1/Dense/Ifran/Phase2/Y2020/Annual/Ifran_B4_C15_Phase2_Y2020_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif",
        "products": ["Our B4 Phase 2", "Lang 2020", "Pauls 2020", "Meta/Tolan 2023", "GFCH 2019"],
    },
    "Maamoura": {
        "key": "Maamoura",
        "ecosystem": "low-sparsity forest",
        "map_year": 2019,
        "eval_max": 20.0,
        "height_edges": [0, 5, 10, 15, 20],
        "catalog": Path(r"C:\Users\Dell\Desktop\Publication_Clarck") / "Data/Low_Sparsity/Maamoura/Temporal_Catalogs/T4_DENSE_2019_2025_C15",
        "ours": PROJECT / "Inference_Harmonized_GEDIAnchored_NaturalP1/Low_Sparsity/Maamoura/Phase2/Y2019/Annual/Maamoura_B4_C15_Phase2_Y2019_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif",
        "products": ["Our B4 Phase 2", "Lang 2020", "Pauls 2020", "Meta/Tolan 2023", "GFCH 2019"],
    },
    "Agadir": {
        "key": "Agadir",
        "ecosystem": "sparse forest",
        "map_year": 2020,
        "eval_max": 20.0,
        "height_edges": [0, 5, 10, 15, 20],
        "catalog": Path(r"C:\Users\Dell\Desktop\Publication_Clarck") / "Data/Sparse/Agadir/Catalogs/final_catalog",
        "ours": PROJECT / "Inference_Harmonized_GEDIAnchored_NaturalP1/Sparse/Agadir/Phase2/Y2020/Annual/Agadir_B4_C15_Phase2_Y2020_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif",
        "products": ["Our B4 Phase 2", "Lang 2020", "Pauls 2020", "Meta/Tolan 2023"],
    },
}

def annual_map_path(cfg, year):
    """Derive a Phase-2 annual-map path without silently reusing another year."""
    nominal_token = f"Y{int(cfg['map_year'])}"
    target_token = f"Y{int(year)}"
    original = str(cfg["ours"])
    derived = original.replace(
        f"\\{nominal_token}\\", f"\\{target_token}\\"
    ).replace(
        f"_{nominal_token}_", f"_{target_token}_"
    )
    if derived == original and int(year) != int(cfg["map_year"]):
        raise RuntimeError(
            f"Cannot derive annual map path for year {year} from {original}"
        )
    return Path(derived)


for forest, cfg in SITES.items():
    root = PRODUCT_ROOT / cfg["key"]
    pauls_crs_tag = "EPSG32630" if forest == "Ifran" else "EPSG32629"
    pauls_root = (
        PRODUCT_ROOT / "_PAULS_2020_VERIFIED_V1" / forest
        / "Pauls_et_al_2024_CHM_2020_10m" / "clean" / "mosaic"
    )
    cfg["annual_maps"] = {
        year: annual_map_path(cfg, year) for year in ANNUAL_MAP_YEARS
    }
    cfg["maps"] = {
        # This fixed path is retained only for backwards-compatible metadata.
        # Sampling Our Model always uses cfg["annual_maps"][gedi_year].
        "Our B4 Phase 2": cfg["ours"],
        "Lang 2020": root / f"ETH_Lang_2020_CHM_10m/clean/mosaic/{cfg['key']}__ETH_Lang_2020_CHM_10m__clean__EPSG32630.tif",
        "GFCH 2019": root / f"GFCH_Potapov_GLAD_2019_30m/clean/mosaic/{cfg['key']}__GFCH_Potapov_GLAD_2019_30m__clean__EPSG32630.tif",
        "Meta/Tolan 2023": root / f"Meta_WRI_Tolan_2023_CHM_resampled_10m/clean/mosaic/{cfg['key']}__Meta_WRI_Tolan_2023_CHM_resampled_10m__clean__EPSG32630.tif",
        "Pauls 2020": pauls_root / f"{forest}__Pauls_et_al_2024_CHM_2020_10m__clean__{pauls_crs_tag}.tif",
    }

required = []
for cfg in SITES.values():
    required.append(cfg["catalog"] / "shot_catalog_step05.csv.gz")
    required.extend(cfg["annual_maps"].values())
    required.extend(
        cfg["maps"][product]
        for product in cfg["products"]
        if product != OUR_PRODUCT
    )
missing = [str(path) for path in required if not Path(path).is_file()]
if missing:
    raise FileNotFoundError("Missing comparison inputs:\n" + "\n".join(missing))

# PAULS_NUMERIC_PREFLIGHT_V1 — reject stale all-NoData files from the old unit bug.
for forest, cfg in SITES.items():
    pauls_path = Path(cfg["maps"]["Pauls 2020"])
    with rasterio.open(pauls_path) as source_raster:
        array = source_raster.read(1).astype(np.float64)
        valid = np.isfinite(array) & (array != -9999.0)
        if source_raster.nodata is not None and np.isfinite(source_raster.nodata):
            valid &= array != source_raster.nodata
        values = array[valid]
        if values.size == 0:
            raise RuntimeError(
                f"{forest}: Pauls raster is all-NoData. Re-run "
                "script_Download_CHM_global_Product.ipynb after the ×0.01 unit fix: "
                f"{pauls_path}"
            )
        if float(values.min()) < 0.0 or float(values.max()) > 80.0:
            raise RuntimeError(
                f"{forest}: Pauls values are not physical metre heights: "
                f"min={values.min()}, max={values.max()}, path={pauls_path}"
            )
        rx, ry = map(abs, source_raster.res)
        if not (9.95 <= rx <= 10.05 and 9.95 <= ry <= 10.05):
            raise RuntimeError(
                f"{forest}: Pauls resolution must be 10 m, got {(rx, ry)}"
            )
    print(
        f"[PAULS PASS] {forest}: n={values.size:,}, "
        f"range={values.min():.2f}–{values.max():.2f} m, res=10 m"
    )

print(f"Preflight PASS — {len(required)} required inputs are available.")
print("Output directory:", OUT_DIR)


In [ ]:
def load_test_points(forest, cfg):
    shots = pd.read_csv(cfg["catalog"] / "shot_catalog_step05.csv.gz", low_memory=False)
    shots = shots[shots["split"].astype(str).str.lower().eq("test")].copy()
    shots["rh95"] = pd.to_numeric(shots["rh95"], errors="coerce")
    shots = shots[shots["rh95"].between(2.0, cfg["eval_max"])].copy()
    shots["gedi_year"] = pd.to_datetime(shots["aux_gedi_date"], errors="coerce").dt.year

    if {"aux_lon", "aux_lat"}.issubset(shots.columns):
        shots["lon"] = pd.to_numeric(shots["aux_lon"], errors="coerce")
        shots["lat"] = pd.to_numeric(shots["aux_lat"], errors="coerce")
    else:
        if forest != "Ifran":
            raise KeyError(f"{forest}: aux_lon/aux_lat are missing from the TEST catalog.")
        coordinate_file = Path(r"E:\CHM\Ifran_6\DATA\GEDI\Output_Preproc_L2A\GEDI_QC_ACQ_clean.csv.gz")
        coordinates = pd.read_csv(coordinate_file, usecols=["shot_number", "lon", "lat"])
        coordinates["shot_number"] = pd.to_numeric(coordinates["shot_number"], errors="coerce").astype("Int64")
        shots["aux_shot_id"] = pd.to_numeric(shots["aux_shot_id"], errors="coerce").astype("Int64")
        shots = shots.merge(
            coordinates, left_on="aux_shot_id", right_on="shot_number",
            how="left", validate="many_to_one",
        )

    shots = shots.dropna(subset=["lon", "lat", "rh95", "gedi_year"])
    shots = (
        shots.sort_values(
            ["aux_shot_uid", "aux_abs_temporal_delta_days", "aux_gedi_date"],
            kind="stable",
        )
        .drop_duplicates("aux_shot_uid", keep="first")
    )
    shots["shot_id"] = shots["aux_shot_uid"].astype(str)
    return shots[["shot_id", "rh95", "gedi_year", "lon", "lat"]]


def footprint_mean(source, x, y):
    footprint = Point(float(x), float(y)).buffer(RADIUS_M, quad_segs=24)
    raw = from_bounds(*footprint.bounds, transform=source.transform)
    col0 = max(0, int(math.floor(raw.col_off)) - 1)
    row0 = max(0, int(math.floor(raw.row_off)) - 1)
    col1 = min(source.width, int(math.ceil(raw.col_off + raw.width)) + 1)
    row1 = min(source.height, int(math.ceil(raw.row_off + raw.height)) + 1)
    if col1 <= col0 or row1 <= row0:
        return np.nan, 0.0

    window = Window(col0, row0, col1 - col0, row1 - row0)
    array = source.read(1, window=window, masked=True)
    values = np.ma.filled(array, np.nan).astype(float)
    masked = np.ma.getmaskarray(array)
    local_rows, local_cols = np.indices(values.shape)
    rows, cols = local_rows + row0, local_cols + col0
    affine = source.transform
    xa, xb = affine.c + cols * affine.a, affine.c + (cols + 1) * affine.a
    ya, yb = affine.f + rows * affine.e, affine.f + (rows + 1) * affine.e
    pixels = shapely.box(
        np.minimum(xa, xb), np.minimum(ya, yb),
        np.maximum(xa, xb), np.maximum(ya, yb),
    )
    areas = shapely.area(shapely.intersection(pixels, footprint))
    valid = (
        (areas > 1e-9) & (~masked) & np.isfinite(values)
        & (values >= 0) & (values <= 100)
    )
    coverage = float(areas[valid].sum() / footprint.area)
    if coverage < MIN_COVERAGE or not valid.any():
        return np.nan, coverage
    prediction = float(np.sum(values[valid] * areas[valid]) / areas[valid].sum())
    return prediction, coverage


def sample_product(points, product, path):
    predictions = np.full(len(points), np.nan, dtype=np.float32)
    coverages = np.zeros(len(points), dtype=np.float32)
    with rasterio.open(path) as source:
        xs, ys = transform(
            "EPSG:4326", source.crs,
            points["lon"].tolist(), points["lat"].tolist(),
        )
        for index, (x, y) in enumerate(zip(xs, ys)):
            predictions[index], coverages[index] = footprint_mean(source, x, y)

    result = points.copy()
    result["product"] = product
    result["prediction"] = predictions
    result["coverage"] = coverages
    result["error"] = result["prediction"] - result["rh95"]
    return result[np.isfinite(result["prediction"])].copy()


def sample_our_model_year_matched(points, cfg):
    """Sample each GEDI TEST shot from Our Model map of the same calendar year."""
    parts = []
    observed_years = sorted(points["gedi_year"].astype(int).unique())
    unavailable = sorted(set(observed_years) - set(cfg["annual_maps"]))
    if unavailable:
        raise RuntimeError(f"Missing annual-map registry entries: {unavailable}")

    for year in observed_years:
        annual_path = cfg["annual_maps"][int(year)]
        if not annual_path.is_file():
            raise FileNotFoundError(
                f"Annual Phase-2 map missing for GEDI year {year}: {annual_path}"
            )
        year_points = points[points["gedi_year"].astype(int).eq(int(year))].copy()
        sampled = sample_product(
            year_points, OUR_PRODUCT, annual_path
        )
        sampled["product_year"] = int(year)
        sampled["temporal_protocol"] = "exact GEDI-year to annual-map match"
        parts.append(sampled)

    result = pd.concat(parts, ignore_index=True)
    if not (
        result["gedi_year"].astype(int)
        == result["product_year"].astype(int)
    ).all():
        raise AssertionError("Our Model contains a GEDI/map year mismatch")
    return result


def aggregate_forest(forest, cfg):
    """Protocol A: all canonical TEST years; Our Model is exactly year-matched."""
    points = load_test_points(forest, cfg)
    sampled = [
        sample_our_model_year_matched(points, cfg).assign(forest=forest)
    ]
    for product in cfg["products"]:
        if product == OUR_PRODUCT:
            continue
        part = sample_product(points, product, cfg["maps"][product])
        part["product_year"] = PRODUCT_NOMINAL_YEARS[product]
        part["temporal_protocol"] = "static product on multiannual canonical TEST"
        sampled.append(part.assign(forest=forest))
    return pd.concat(sampled, ignore_index=True)


CACHE = OUT_DIR / "GEDI_TEST_product_valid_support_signed_errors.csv.gz"
CACHE_METADATA = OUT_DIR / "GEDI_TEST_product_valid_support_cache_metadata.json"


def comparison_input_fingerprint():
    import hashlib

    inputs = []
    protocol = {
        "radius_m": RADIUS_M,
        "minimum_coverage": MIN_COVERAGE,
        "deduplication": "unique-nearest",
        "evaluation_min_m": 2.0,
        "support_policy": "all canonical TEST years; Our Model exactly year-matched; static CHMs multiannual",
        "forests": {},
    }
    for forest, cfg in SITES.items():
        paths = [cfg["catalog"] / "shot_catalog_step05.csv.gz"]
        for annual_path in cfg["annual_maps"].values():
            manifest_path = inference_manifest_path(annual_path)
            if not manifest_path.is_file():
                raise FileNotFoundError(f"{forest}: missing inference lineage {manifest_path}")
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            if manifest.get("phase1_sha256") != PHASE1_EXPECTED_SHA256[forest]:
                raise RuntimeError(f"{forest}: wrong Natural Sampling Phase 1 lineage")
            phase2_checkpoint = Path(manifest.get("phase2_checkpoint", ""))
            selected_registry = json.loads((PROJECT / "Source" / "Project" / "final_selected_phase2_models.json").read_text(encoding="utf-8"))["models"]
            selected = selected_registry[forest.lower()]
            if manifest.get("phase2_sha256") != selected["checkpoint_sha256"]:
                raise RuntimeError(f"{forest}: map Phase 2 hash differs from the final registry")
            paths.extend([annual_path, manifest_path])
        paths.extend(
            cfg["maps"][product]
            for product in cfg["products"]
            if product != OUR_PRODUCT
        )
        protocol["forests"][forest] = {
            "evaluation_max_m": cfg["eval_max"],
            "products": cfg["products"],
            "annual_our_model_maps": {
                str(year): str(path.resolve())
                for year, path in cfg["annual_maps"].items()
            },
            "paths": [str(path.resolve()) for path in paths],
        }
        for path in paths:
            stat = path.stat()
            inputs.append({
                "path": str(path.resolve()),
                "size": stat.st_size,
                "mtime_ns": stat.st_mtime_ns,
            })
    payload = {"protocol": protocol, "inputs": inputs}
    digest = hashlib.sha256(
        json.dumps(payload, sort_keys=True).encode("utf-8")
    ).hexdigest()
    return digest, payload


current_fingerprint, fingerprint_payload = comparison_input_fingerprint()
cache_is_current = False
if CACHE.is_file() and CACHE_METADATA.is_file():
    metadata = json.loads(CACHE_METADATA.read_text(encoding="utf-8"))
    cache_is_current = metadata.get("fingerprint") == current_fingerprint

if RECOMPUTE_AGGREGATION or not cache_is_current:
    if CACHE.is_file() and not cache_is_current:
        print("Cache rejected: input/protocol fingerprint is outdated.")
    parts = []
    for forest, cfg in SITES.items():
        print(f"Sampling every canonical TEST shot for {forest} ...")
        parts.append(aggregate_forest(forest, cfg))
    valid_samples = pd.concat(parts, ignore_index=True)
    valid_samples.to_csv(CACHE, index=False, compression="gzip")
    CACHE_METADATA.write_text(
        json.dumps(
            {"fingerprint": current_fingerprint, "payload": fingerprint_payload},
            indent=2,
        ),
        encoding="utf-8",
    )
else:
    valid_samples = pd.read_csv(CACHE, low_memory=False)
    print("Reused fingerprint-validated product-support table:", CACHE)

expected_forests = set(SITES)
if set(valid_samples["forest"]) != expected_forests:
    raise RuntimeError(
        f"Cache forest mismatch: {set(valid_samples['forest'])} != {expected_forests}."
    )


our_temporal_audit = valid_samples[
    valid_samples["product"].eq(OUR_PRODUCT)
].copy()
if our_temporal_audit.empty:
    raise RuntimeError("Our Model year-matched samples are absent")
if not (
    pd.to_numeric(our_temporal_audit["gedi_year"]).astype(int)
    == pd.to_numeric(our_temporal_audit["product_year"]).astype(int)
).all():
    mismatch = our_temporal_audit.loc[
        pd.to_numeric(our_temporal_audit["gedi_year"]).astype(int)
        != pd.to_numeric(our_temporal_audit["product_year"]).astype(int),
        ["forest", "shot_id", "gedi_year", "product_year"],
    ]
    raise AssertionError(
        "Our Model temporal guard failed:\n" + mismatch.head().to_string(index=False)
    )
print("[PASS] Our Model: every valid GEDI TEST shot uses the map of the same year.")


def exact_product_ids(frame, product):
    return frozenset(
        frame.loc[frame["product"].eq(product), "shot_id"].astype(str)
    )


common_parts = []
pair_parts = []
support_rows = []
pair_rows = []

for forest, cfg in SITES.items():
    forest_samples = valid_samples[valid_samples["forest"].eq(forest)].copy()
    present_products = set(forest_samples["product"])
    if present_products != set(cfg["products"]):
        raise RuntimeError(
            f"{forest}: cached products {present_products} != {set(cfg['products'])}."
        )

    canonical_n = int(load_test_points(forest, cfg)["shot_id"].nunique())
    product_id_sets = {
        product: exact_product_ids(forest_samples, product)
        for product in cfg["products"]
    }

    # Strict all-product common support is retained only for the boxplot.
    all_common_ids = set.intersection(
        *(set(identifiers) for identifiers in product_id_sets.values())
    )
    if len(all_common_ids) < 30:
        raise RuntimeError(f"{forest}: insufficient all-product common support.")
    for product in cfg["products"]:
        part = forest_samples[
            forest_samples["product"].eq(product)
            & forest_samples["shot_id"].astype(str).isin(all_common_ids)
        ].copy()
        if frozenset(part["shot_id"].astype(str)) != frozenset(all_common_ids):
            raise AssertionError(f"{forest}/{product}: boxplot shot_id mismatch.")
        common_parts.append(part)

    # Each scatter pair uses its own maximal valid intersection.
    ours_ids = product_id_sets["Our B4 Phase 2"]
    for competitor in cfg["products"][1:]:
        pair_ids = ours_ids.intersection(product_id_sets[competitor])
        if len(pair_ids) < 30:
            raise RuntimeError(
                f"{forest}/{competitor}: insufficient pairwise support (n={len(pair_ids)})."
            )
        pair_id = f"{forest} | Our B4 Phase 2 vs {competitor}"
        for product in ("Our B4 Phase 2", competitor):
            part = forest_samples[
                forest_samples["product"].eq(product)
                & forest_samples["shot_id"].astype(str).isin(pair_ids)
            ].copy()
            part["pair_id"] = pair_id
            part["competitor"] = competitor
            if frozenset(part["shot_id"].astype(str)) != pair_ids:
                raise AssertionError(f"{pair_id}/{product}: exact shot_id mismatch.")
            pair_parts.append(part)
        pair_rows.append({
            "forest": forest,
            "pair_id": pair_id,
            "canonical_test_n": canonical_n,
            "our_valid_n": len(ours_ids),
            "competitor": competitor,
            "competitor_valid_n": len(product_id_sets[competitor]),
            "pairwise_common_n": len(pair_ids),
        })

    support_rows.append({
        "forest": forest,
        "ecosystem": cfg["ecosystem"],
        "evaluation_domain_m": f"2–{cfg['eval_max']:g}",
        "canonical_test_n": canonical_n,
        "footprint_diameter_m": 2 * RADIUS_M,
        "minimum_valid_coverage": MIN_COVERAGE,
        "all_product_common_n_for_boxplot": len(all_common_ids),
        **{
            f"{product}_valid_n": len(product_id_sets[product])
            for product in cfg["products"]
        },
    })

common_errors = pd.concat(common_parts, ignore_index=True)
paired_errors = pd.concat(pair_parts, ignore_index=True)
support_audit = pd.DataFrame(support_rows)
pairwise_support_audit = pd.DataFrame(pair_rows)

support_audit.to_csv(OUT_DIR / "TEST_support_audit.csv", index=False)
pairwise_support_audit.to_csv(
    OUT_DIR / "pairwise_TEST_support_audit.csv", index=False
)
paired_errors.to_csv(
    OUT_DIR / "pairwise_TEST_signed_errors.csv.gz",
    index=False,
    compression="gzip",
)
display(support_audit)
display(pairwise_support_audit)


# Chapter 1 — Ifran dense forest

The Ifran comparison covers GEDI RH95 from 2 to 45 m. The first displayed
class is 0–5 m and therefore contains the retained 2–5 m TEST observations.
All four CHMs are evaluated on the same area-weighted GEDI TEST footprints.

The chapter reports both the signed-error distributions by height class and GEDI-versus-CHM scatter plots on that identical common support.

The publication overview aligns all available CHM scatter panels in one row using identical axes and a shared concentration scale. Its panels use each product's maximum valid TEST support; the paired figures retain identical shot support for formal product-to-product comparison.


In [ ]:
def legend_label(product, cfg):
    if product == "Our B4 Phase 2":
        return "Our Model annual (year-matched)"
    return PRODUCT_LEGENDS[product]


def plot_forest_comparison(forest):
    cfg = SITES[forest]
    common_plot = common_errors[common_errors["forest"].eq(forest)].copy()
    edges = np.asarray(cfg["height_edges"], dtype=float)
    labels = [f"{edges[index]:g}–{edges[index + 1]:g} m" for index in range(len(edges) - 1)]
    common_plot["height_bin"] = pd.cut(
        common_plot["rh95"],
        bins=edges,
        labels=labels,
        include_lowest=True,
        right=True,
    )

    products = cfg["products"]
    centers = np.arange(len(labels), dtype=float)
    total_width = 0.82
    box_width = total_width / max(len(products), 1)
    common_n = int(common_plot["shot_id"].nunique())

    fig, ax = plt.subplots(figsize=(15, 6.5))
    for product_index, product in enumerate(products):
        series = [
            common_plot.loc[
                common_plot["product"].eq(product)
                & common_plot["height_bin"].astype(str).eq(label),
                "error",
            ].dropna().to_numpy()
            for label in labels
        ]
        positions = (
            centers - total_width / 2 + box_width / 2
            + product_index * box_width
        )
        boxplot = ax.boxplot(
            series,
            positions=positions,
            widths=box_width * 0.9,
            patch_artist=True,
            showfliers=False,
            whis=1.5,
            medianprops={"color": "black", "linewidth": 1.0},
            whiskerprops={"color": "0.45", "linewidth": 0.8},
            capprops={"color": "0.45", "linewidth": 0.8},
            manage_ticks=False,
        )
        for patch in boxplot["boxes"]:
            patch.set_facecolor(COLORS[product])
            patch.set_edgecolor("0.4")

    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_xticks(centers)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_xlabel("GEDI RH95 height class (m)")
    ax.set_ylabel("Signed error (CHM \u2212 GEDI RH95, m)")
    ax.set_title(
        f"{forest} — signed-error distributions by height class\n"
        f"Strictly common GEDI TEST support: n={common_n:,}; "
        "Our Model is year-matched; static CHM nominal years differ",
        fontweight="bold",
    )
    ax.grid(True, axis="y", alpha=0.2)
    ax.legend(
        handles=[
            Patch(
                facecolor=COLORS[product],
                edgecolor="0.4",
                label=legend_label(product, cfg),
            )
            for product in products
        ],
        loc="lower left",
        ncol=2,
    )
    fig.tight_layout()

    stem = f"{forest}_signed_error_by_height_class"
    saved = []
    for extension in ("png", "svg", "pdf"):
        destination = OUT_DIR / f"{stem}.{extension}"
        fig.savefig(
            destination,
            dpi=600 if extension == "png" else None,
            bbox_inches="tight",
            facecolor="white",
        )
        saved.append(destination)
    plt.show()
    plt.close(fig)
    print("Saved:")
    for destination in saved:
        print(" -", destination)



# Standalone imports make this visualization block safe after a kernel restart.
import re
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize


def scatter_metrics(df: pd.DataFrame) -> dict:
    observed = df["rh95"].to_numpy(float)
    predicted = df["prediction"].to_numpy(float)
    valid = np.isfinite(observed) & np.isfinite(predicted)
    observed, predicted = observed[valid], predicted[valid]
    residual = predicted - observed
    observed_std = float(np.std(observed, ddof=0))
    predicted_std = float(np.std(predicted, ddof=0))
    if len(observed) >= 2 and observed_std > 0 and predicted_std > 0:
        correlation = float(np.corrcoef(observed, predicted)[0, 1])
        slope = float(
            np.cov(observed, predicted, ddof=0)[0, 1]
            / np.var(observed)
        )
    else:
        correlation, slope = np.nan, np.nan
    total_sum_squares = float(np.sum((observed - np.mean(observed)) ** 2))
    residual_sum_squares = float(np.sum(residual ** 2))
    predictive_r2 = (
        1.0 - residual_sum_squares / total_sum_squares
        if total_sum_squares > 0
        else np.nan
    )
    return {
        "n": int(len(observed)),
        "mae": float(np.mean(np.abs(residual))),
        "rmse": float(np.sqrt(np.mean(residual ** 2))),
        "bias": float(np.mean(residual)),
        "std_ratio": (
            float(predicted_std / observed_std)
            if observed_std > 0 else np.nan
        ),
        "r2": float(predictive_r2),
        "slope": slope,
        "correlation": correlation,
        "corr": correlation,
    }


def point_concentration_1m(df: pd.DataFrame, axis_max: float) -> np.ndarray:
    """Number of observations in each 1 × 1 m GEDI–CHM cell."""
    x = df["rh95"].to_numpy(float)
    y = df["prediction"].to_numpy(float)
    density = np.ones(len(df), dtype=float)
    visible = (
        np.isfinite(x) & np.isfinite(y)
        & (x >= 0) & (x <= axis_max)
        & (y >= 0) & (y <= axis_max)
    )
    edges = np.arange(0.0, axis_max + 1.0, 1.0)
    if edges[-1] <= axis_max:
        edges = np.append(edges, axis_max + 1.0)
    counts, _, _ = np.histogram2d(x[visible], y[visible], bins=(edges, edges))
    ix = np.clip(
        np.searchsorted(edges, x[visible], side="right") - 1,
        0,
        len(edges) - 2,
    )
    iy = np.clip(
        np.searchsorted(edges, y[visible], side="right") - 1,
        0,
        len(edges) - 2,
    )
    density[visible] = counts[ix, iy]
    return density


def nice_colorbar_scale(max_value: float, target_intervals: int = 6):
    if not np.isfinite(max_value) or max_value <= 0:
        return 1.0, np.array([0, 1], dtype=int)
    raw_step = max_value / target_intervals
    magnitude = 10.0 ** np.floor(np.log10(raw_step))
    fraction = raw_step / magnitude
    if fraction <= 1.0:
        nice_fraction = 1.0
    elif fraction <= 2.0:
        nice_fraction = 2.0
    elif fraction <= 5.0:
        nice_fraction = 5.0
    else:
        nice_fraction = 10.0
    step = max(1.0, nice_fraction * magnitude)
    upper = float(np.ceil(max_value / step) * step)
    ticks = np.arange(0.0, upper + 0.5 * step, step)
    return upper, np.rint(ticks).astype(int)


def scatter_panel(ax, frame, title, density, density_norm, axis_max):
    metrics = scatter_metrics(frame)
    order = np.argsort(density, kind="mergesort")
    points = ax.scatter(
        frame["rh95"].to_numpy(float)[order],
        frame["prediction"].to_numpy(float)[order],
        c=density[order],
        cmap="viridis",
        norm=density_norm,
        s=17,
        marker="o",
        edgecolors="none",
        alpha=0.78,
        rasterized=True,
        zorder=2,
    )
    ax.plot(
        [0, axis_max], [0, axis_max],
        "k--", linewidth=1.2, label="1:1", zorder=3,
    )
    ax.set_xlim(0, axis_max)
    ax.set_ylim(0, axis_max)
    ax.set_aspect("equal", adjustable="box")
    ax.set_box_aspect(1)
    ax.set_anchor("C")
    ax.set_title(title)
    ax.set_xlabel("GEDI RH95 height class (m)")
    ax.set_ylabel("Signed error (CHM \u2212 GEDI RH95, m)")
    ax.legend(loc="upper left", frameon=True)
    ax.text(
        0.97,
        0.04,
        (
            f"n={metrics['n']:,}\n"
            f"MAE={metrics['mae']:.2f} m\n"
            f"RMSE={metrics['rmse']:.2f} m\n"
            f"Bias={metrics['bias']:+.2f} m\n"
            f"R²={metrics['r2']:.2f}\n"
            f"Slope={metrics['slope']:.2f}"
        ),
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        bbox={"facecolor": "white", "alpha": 0.86, "edgecolor": "0.75"},
    )
    return points, metrics


def plot_forest_pairwise_scatters(forest):
    cfg = SITES[forest]
    forest_pairs = paired_errors[paired_errors["forest"].eq(forest)]
    metric_rows = []

    for pair_id, pair_frame in forest_pairs.groupby("pair_id", sort=False):
        products = list(pair_frame["product"].drop_duplicates())
        density_by_product = {
            product: point_concentration_1m(
                pair_frame[pair_frame["product"].eq(product)],
                cfg["eval_max"],
            )
            for product in products
        }
        max_density = max(
            float(np.max(values)) for values in density_by_product.values()
        )
        colorbar_upper, colorbar_ticks = nice_colorbar_scale(max_density)
        shared_norm = Normalize(vmin=0.0, vmax=colorbar_upper)

        figure, axes = plt.subplots(
            1,
            len(products),
            figsize=(7.15 * len(products), 6.8),
            sharex=True,
            sharey=True,
            constrained_layout=True,
            gridspec_kw={"width_ratios": [1] * len(products)},
        )
        axes = np.atleast_1d(axes)
        mappable = None
        for axis, product in zip(axes, products):
            part = pair_frame[pair_frame["product"].eq(product)]
            mappable, metrics = scatter_panel(
                axis,
                part,
                legend_label(product, cfg),
                density_by_product[product],
                shared_norm,
                cfg["eval_max"],
            )
            metric_rows.append({
                "forest": forest,
                "pair_id": pair_id,
                "product": product,
                **metrics,
            })

        colorbar = figure.colorbar(
            mappable,
            ax=axes.tolist(),
            location="right",
            pad=0.025,
            shrink=1.0,
            aspect=35,
            anchor=(0.0, 0.5),
        )
        colorbar.set_ticks(colorbar_ticks)
        colorbar.set_ticklabels([f"{tick:d}" for tick in colorbar_ticks])
        colorbar.set_label("Number of samples")
        pair_n = int(pair_frame["shot_id"].nunique())
        competitor = pair_frame["competitor"].iloc[0]
        figure.suptitle(
            f"{forest} — paired GEDI TEST validation\n"
            f"Our B4 Phase 2 versus {legend_label(competitor, cfg)} — n={pair_n:,}",
            fontsize=15,
            fontweight="bold",
        )

        safe_competitor = re.sub(r"[^A-Za-z0-9]+", "_", competitor).strip("_")
        stem = f"{forest}_paired_scatter_Our_B4_vs_{safe_competitor}"
        for extension in ("png", "svg", "pdf"):
            destination = OUT_DIR / f"{stem}.{extension}"
            figure.savefig(
                destination,
                dpi=600 if extension == "png" else None,
                bbox_inches="tight",
                facecolor="white",
            )
            print("Saved:", destination)
        plt.show()
        plt.close(figure)

    metrics_table = pd.DataFrame(metric_rows)
    metrics_table.to_csv(
        OUT_DIR / f"{forest}_pairwise_scatter_metrics.csv",
        index=False,
    )
    display(metrics_table)


def plot_forest_scatter_overview(forest):
    """Publication overview: one row, one panel per available CHM product."""
    cfg = SITES[forest]
    forest_samples = valid_samples[valid_samples["forest"].eq(forest)].copy()
    products = cfg["products"]
    canonical_n = int(
        support_audit.loc[
            support_audit["forest"].eq(forest), "canonical_test_n"
        ].iloc[0]
    )

    density_by_product = {
        product: point_concentration_1m(
            forest_samples[forest_samples["product"].eq(product)],
            cfg["eval_max"],
        )
        for product in products
    }
    maximum_density = max(
        float(np.max(values)) for values in density_by_product.values()
    )
    colorbar_upper, colorbar_ticks = nice_colorbar_scale(maximum_density)
    shared_norm = Normalize(vmin=0.0, vmax=colorbar_upper)

    # Explicit GridSpec: larger square panels and a dedicated colorbar axis.
    # The colorbar therefore has exactly the same top and bottom as the X/Y axes.
    n_products = len(products)
    figure = plt.figure(
        figsize=(5.8 * n_products + 0.8, 7.2),
        constrained_layout=False,
    )
    grid = figure.add_gridspec(
        1,
        n_products + 1,
        width_ratios=[1.0] * n_products + [0.045],
        left=0.055,
        right=0.965,
        bottom=0.115,
        top=0.82,
        wspace=0.08,
    )
    axes = []
    for panel_index in range(n_products):
        if panel_index == 0:
            axis = figure.add_subplot(grid[0, panel_index])
        else:
            axis = figure.add_subplot(
                grid[0, panel_index],
                sharex=axes[0],
                sharey=axes[0],
            )
        axes.append(axis)
    colorbar_axis = figure.add_subplot(grid[0, -1])
    mappable = None
    metric_rows = []

    for panel_index, (axis, product) in enumerate(zip(axes, products)):
        frame = forest_samples[forest_samples["product"].eq(product)].copy()
        mappable, metrics = scatter_panel(
            axis,
            frame,
            legend_label(product, cfg),
            density_by_product[product],
            shared_norm,
            cfg["eval_max"],
        )
        axis.text(
            0.01,
            1.035,
            f"({chr(97 + panel_index)})",
            transform=axis.transAxes,
            ha="left",
            va="bottom",
            fontsize=11,
            fontweight="bold",
        )
        # Publication convention: the 1:1 legend is fixed in the upper-left.
        legend_handles, legend_labels = axis.get_legend_handles_labels()
        axis.legend(
            legend_handles,
            legend_labels,
            loc="upper left",
            frameon=True,
            borderaxespad=0.6,
        )
        axis.set_title(axis.get_title(), fontsize=12, pad=8)
        axis.tick_params(axis="both", labelsize=10)
        if panel_index > 0:
            axis.set_ylabel("Signed error (CHM \u2212 GEDI RH95, m)")
        metric_rows.append({
            "forest": forest,
            "product": product,
            "canonical_test_n": canonical_n,
            "valid_product_support_n": metrics["n"],
            "support_mode": "maximum valid TEST support for this product",
            **metrics,
        })

    colorbar = figure.colorbar(
        mappable,
        cax=colorbar_axis,
    )
    colorbar.set_ticks(colorbar_ticks)
    colorbar.set_ticklabels([f"{tick:d}" for tick in colorbar_ticks])
    colorbar.set_label("Number of samples")
    figure.suptitle(
        f"{forest} — GEDI RH95 versus canopy-height products\n"
        f"Maximum valid TEST support per product; canonical TEST n={canonical_n:,}",
        fontsize=16,
        fontweight="bold",
        y=0.965,
    )

    stem = f"{forest}_all_CHM_scatter_one_row"
    for extension in ("png", "svg", "pdf"):
        destination = OUT_DIR / f"{stem}.{extension}"
        figure.savefig(
            destination,
            dpi=600 if extension == "png" else None,
            bbox_inches="tight",
            facecolor="white",
        )
        print("Saved:", destination)
    plt.show()
    plt.close(figure)

    overview_metrics = pd.DataFrame(metric_rows)
    overview_metrics.to_csv(
        OUT_DIR / f"{forest}_all_CHM_scatter_one_row_metrics.csv",
        index=False,
    )
    display(overview_metrics)




# Chapter 2 — Maamoura low-sparsity forest

The Maamoura comparison covers GEDI RH95 from 2 to 20 m. The first displayed
class is 0–5 m and therefore contains the retained 2–5 m TEST observations.
All four CHMs are evaluated on the same area-weighted GEDI TEST footprints.

The chapter reports both the signed-error distributions by height class and GEDI-versus-CHM scatter plots on that identical common support.

The publication overview aligns all available CHM scatter panels in one row using identical axes and a shared concentration scale. Its panels use each product's maximum valid TEST support; the paired figures retain identical shot support for formal product-to-product comparison.


# Chapter 3 — Agadir sparse forest

The Agadir comparison covers GEDI RH95 from 2 to 20 m. The first displayed
class is 0–5 m and therefore contains the retained 2–5 m TEST observations.
Our B4 Phase 2, Lang 2020 and Meta/Tolan 2023 are evaluated on the same
area-weighted GEDI TEST footprints. GFCH 2019 is omitted because its local
valid coverage is insufficient for a defensible common-support comparison (about 0.14% valid local coverage and only 17 strictly common TEST shots).

The chapter reports both the signed-error distributions by height class and GEDI-versus-CHM scatter plots on that identical common support.

The publication overview aligns all available CHM scatter panels in one row using identical axes and a shared concentration scale. Its panels use each product's maximum valid TEST support; the paired figures retain identical shot support for formal product-to-product comparison.


# Chapter 4 — Article-ready scatter plots

This chapter generates only the one-row scientific scatter comparison for each forest.

- Our Model is always the left-most panel.
- Every panel reports `n`, Pearson Corr, MAE, RMSE, bias and regression slope in the upper-left corner.
- Axes, concentration scale and colorbar are shared coherently within each forest.
- No signed-error plot and no height-distribution plot are generated.
- The height-stratified Schwartz decomposition table remains restricted to Our Model.

A final journal-style matrix assembles the three ecosystems in rows and the four CHM products in columns. Agadir–GFCH is retained as an explicit `Not evaluable` panel rather than being silently omitted.

An additional experimental matrix uses uniform 0–45 m X/Y axes, with X tick labels only on the bottom row and Y tick labels only on the left column. Both matrix versions are exported so that the most legible article layout can be selected visually.

Product abbreviations follow the first-author/publication-year convention used by Schwartz et al.: L23 for Lang et al. (2023), P21 for Potapov et al. (2021), and T24 for Tolan et al. (2024). These publication years must not be confused with the nominal map years (2020 for L23 and 2019 for P21).


In [ ]:
# STEP — ORDERED ARTICLE SCATTERS + HEIGHT DISTRIBUTIONS (NO SIGNED-ERROR PLOT)
from matplotlib.colors import LogNorm, PowerNorm, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


GEDI_COLOR = "#526D82"
ARTICLE_COLORS = {
    "GEDI TEST": GEDI_COLOR,
    "Our B4 Phase 2": COLORS["Our B4 Phase 2"],
    "Lang 2020": COLORS["Lang 2020"],
    "GFCH 2019": "#2AA889",
    "Meta/Tolan 2023": COLORS["Meta/Tolan 2023"],
}


def export_article_figure(figure, stem):
    destinations = []
    for extension in ("png", "svg", "pdf"):
        destination = OUT_DIR / f"{stem}.{extension}"
        figure.savefig(
            destination,
            dpi=600 if extension == "png" else None,
            bbox_inches="tight",
            facecolor="white",
        )
        destinations.append(destination)
    for destination in destinations:
        print("Saved:", destination)
    return destinations


def extended_schwartz_metrics(frame):
    """RMSE/RMSPE/Bias and the Kobayashi-Salam MSD decomposition."""
    observed = frame["rh95"].to_numpy(float)
    predicted = frame["prediction"].to_numpy(float)
    valid = np.isfinite(observed) & np.isfinite(predicted) & (observed > 0)
    observed = observed[valid]
    predicted = predicted[valid]
    residual = predicted - observed
    n = int(observed.size)
    if n == 0:
        return {
            "n": 0, "rmse_m": np.nan, "rmspe_pct": np.nan,
            "bias_m": np.nan, "lcs_m2": np.nan, "sdsd_m2": np.nan,
        }

    bias = float(np.mean(residual))
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    rmspe = float(100.0 * np.sqrt(np.mean((residual / observed) ** 2)))
    observed_sd = float(np.std(observed, ddof=0))
    predicted_sd = float(np.std(predicted, ddof=0))

    if n >= 2 and observed_sd > 0 and predicted_sd > 0:
        correlation = float(np.corrcoef(observed, predicted)[0, 1])
        sdsd = float((predicted_sd - observed_sd) ** 2)
        lcs = float(2.0 * predicted_sd * observed_sd * (1.0 - correlation))
        reconstructed_mse = bias ** 2 + sdsd + lcs
        observed_mse = rmse ** 2
        tolerance = max(1e-9, 1e-8 * max(1.0, observed_mse))
        if not np.isclose(observed_mse, reconstructed_mse, rtol=1e-8, atol=tolerance):
            raise AssertionError(
                "MSD decomposition failed: "
                f"MSE={observed_mse:.12g}, Bias²+SDSD+LCS={reconstructed_mse:.12g}"
            )
    else:
        sdsd = np.nan
        lcs = np.nan

    return {
        "n": n,
        "rmse_m": rmse,
        "rmspe_pct": rmspe,
        "bias_m": bias,
        "lcs_m2": lcs,
        "sdsd_m2": sdsd,
    }


def mean_prediction_curve(frame, edges):
    bins = pd.cut(
        frame["rh95"],
        bins=edges,
        include_lowest=True,
        right=True,
    )
    grouped = frame.assign(_bin=bins).groupby("_bin", observed=False)
    summary = grouped.agg(
        observed_mean=("rh95", "mean"),
        predicted_mean=("prediction", "mean"),
        n=("prediction", "size"),
    )
    return summary[summary["n"] >= 3]


def format_metric_value(metric, value):
    if not np.isfinite(value):
        return "NA"
    if metric == "Count":
        return f"{int(round(value)):,}"
    return f"{value:.2f}"


def build_height_class_statistics(forest):
    cfg = SITES[forest]
    # Article table: metrics are intentionally reported only for our model.
    products = ["Our B4 Phase 2"]
    edges = np.asarray(cfg["height_edges"], dtype=float)
    labels = [
        f"{edges[index]:g}–{edges[index + 1]:g} m"
        for index in range(len(edges) - 1)
    ]
    rows = []
    forest_samples = valid_samples[valid_samples["forest"].eq(forest)]

    for product in products:
        part = forest_samples[forest_samples["product"].eq(product)].copy()
        part["height_class"] = pd.cut(
            part["rh95"],
            bins=edges,
            labels=labels,
            include_lowest=True,
            right=True,
        )
        for label in labels:
            class_part = part[part["height_class"].astype(str).eq(label)]
            metric = extended_schwartz_metrics(class_part)
            rows.append({
                "forest": forest,
                "product": product,
                "product_label": legend_label(product, cfg),
                "height_class": label,
                "height_min_m": float(edges[labels.index(label)]),
                "height_max_m": float(edges[labels.index(label) + 1]),
                **metric,
            })

    table = pd.DataFrame(rows)
    expected = len(products) * len(labels)
    if len(table) != expected:
        raise AssertionError((forest, len(table), expected))
    return table


def plot_height_class_metric_table(forest, class_table):
    cfg = SITES[forest]
    # Keep the Schwartz-style table focused on Our B4 Phase 2.
    products = ["Our B4 Phase 2"]
    labels = list(class_table["height_class"].drop_duplicates())
    metric_specs = [
        ("RMSE (m)", "rmse_m"),
        ("RMSPE (%)", "rmspe_pct"),
        ("Bias (m)", "bias_m"),
        ("LCS (m²)", "lcs_m2"),
        ("SDSD (m²)", "sdsd_m2"),
        ("Count", "n"),
    ]

    values = []
    row_labels = []
    row_colors = []
    for product in products:
        product_table = class_table[class_table["product"].eq(product)]
        for metric_label, column in metric_specs:
            indexed = product_table.set_index("height_class")[column]
            values.append([
                format_metric_value(
                    "Count" if metric_label == "Count" else metric_label,
                    float(indexed.get(label, np.nan)),
                )
                for label in labels
            ])
            row_labels.append(metric_label)
            row_colors.append(ARTICLE_COLORS[product])

    figure_height = max(7.0, 0.39 * len(row_labels) + 2.0)
    figure, axis = plt.subplots(figsize=(15.5, figure_height))
    axis.axis("off")
    table_artist = axis.table(
        cellText=values,
        rowLabels=row_labels,
        colLabels=labels,
        cellLoc="center",
        rowLoc="right",
        loc="center",
    )
    table_artist.auto_set_font_size(False)
    table_artist.set_fontsize(8.2)
    table_artist.scale(1.0, 1.32)

    for row_index, color in enumerate(row_colors, start=1):
        if (row_index, -1) in table_artist.get_celld():
            table_artist[(row_index, -1)].set_facecolor(color)
            table_artist[(row_index, -1)].set_alpha(0.25)
        for column_index in range(len(labels)):
            cell = table_artist[(row_index, column_index)]
            cell.set_facecolor(color)
            cell.set_alpha(0.07 if row_index % 2 else 0.11)

    axis.set_title(
        f"{forest} — height-stratified TEST statistics\n"
        "Our Model annual year-matched on its maximum valid canonical TEST support",
        fontsize=14,
        fontweight="bold",
        pad=14,
    )
    figure.tight_layout()
    export_article_figure(figure, f"{forest}_height_class_metrics_table")
    plt.show()
    plt.close(figure)


def plot_ordered_scatter_row(forest):
    cfg = SITES[forest]
    products = list(cfg["products"])
    if products[0] != "Our B4 Phase 2":
        raise AssertionError(f"{forest}: Our B4 Phase 2 must be the first panel")

    forest_samples = valid_samples[valid_samples["forest"].eq(forest)].copy()
    canonical_n = int(
        support_audit.loc[
            support_audit["forest"].eq(forest), "canonical_test_n"
        ].iloc[0]
    )
    density = {}
    for product in products:
        part = forest_samples[forest_samples["product"].eq(product)]
        if part.empty:
            raise AssertionError(f"{forest}: no valid TEST sample for {product}")
        density[product] = point_concentration_1m(part, cfg["eval_max"])

    max_density = max(float(np.nanmax(value)) for value in density.values())
    colorbar_upper, colorbar_ticks = nice_colorbar_scale(max_density)
    shared_norm = Normalize(vmin=0.0, vmax=colorbar_upper)
    n_products = len(products)

    figure = plt.figure(
        figsize=(5.35 * n_products + 0.75, 6.8),
        constrained_layout=False,
    )
    grid = figure.add_gridspec(
        1,
        n_products + 1,
        width_ratios=[1.0] * n_products + [0.045],
        left=0.055,
        right=0.965,
        bottom=0.11,
        top=0.82,
        wspace=0.18,
    )
    axes = []
    rows = []
    mappable = None

    for panel_index, product in enumerate(products):
        axis = (
            figure.add_subplot(grid[0, panel_index])
            if panel_index == 0
            else figure.add_subplot(
                grid[0, panel_index], sharex=axes[0], sharey=axes[0]
            )
        )
        axes.append(axis)
        part = forest_samples[forest_samples["product"].eq(product)].copy()
        order = np.argsort(density[product], kind="mergesort")
        mappable = axis.scatter(
            part["rh95"].to_numpy(float)[order],
            part["prediction"].to_numpy(float)[order],
            c=density[product][order],
            cmap="viridis",
            norm=shared_norm,
            s=14,
            marker="o",
            edgecolors="none",
            alpha=0.78,
            rasterized=True,
            zorder=2,
        )
        axis.plot(
            [0, cfg["eval_max"]],
            [0, cfg["eval_max"]],
            color="0.25",
            linestyle="--",
            linewidth=1.0,
            zorder=3,
        )
        metric = extended_schwartz_metrics(part)
        basic_metric = scatter_metrics(part)
        if product == "Our B4 Phase 2":
            rows.append({
                "forest": forest,
                "product": product,
                "product_label": legend_label(product, cfg),
                "canonical_test_n": canonical_n,
                "support_mode": "maximum valid canonical TEST support",
                **metric,
                "mae_m": basic_metric["mae"],
                "r2": basic_metric["r2"],
                "corr": basic_metric["corr"],
                "slope": basic_metric["slope"],
            })

        axis.set_xlim(0, cfg["eval_max"])
        axis.set_ylim(0, cfg["eval_max"])
        axis.set_aspect("equal", adjustable="box")
        axis.set_box_aspect(1)
        axis.set_title(legend_label(product, cfg), fontsize=11.5, pad=8)
        axis.set_xlabel("GEDI RH95 TEST (m)")
        axis.set_ylabel("CHM height (m)" if panel_index == 0 else "")
        axis.tick_params(
            axis="y",
            which="both",
            left=True,
            labelleft=True,
            right=False,
            labelright=False,
        )
        # Same article metric box for every CHM panel.
        axis.text(
            0.025,
            0.975,
            (
                f"R²={basic_metric['r2']:.2f}\n"
                f"RMSE={basic_metric['rmse']:.2f} m\n"
                f"MAE={basic_metric['mae']:.2f} m\n"
                f"Bias={basic_metric['bias']:+.2f} m\n"
                f"Slope={basic_metric['slope']:.2f}\n"
                f"Corr={basic_metric['corr']:.2f}\n"
                f"n={basic_metric['n']:,}"
            ),
            transform=axis.transAxes,
            ha="left",
            va="top",
            fontsize=9,
            bbox={"facecolor": "white", "alpha": 0.88, "edgecolor": "0.75"},
            zorder=5,
        )
        axis.text(
            0.01,
            1.035,
            f"({chr(97 + panel_index)})",
            transform=axis.transAxes,
            ha="left",
            va="bottom",
            fontsize=11,
            fontweight="bold",
        )

    colorbar_axis = figure.add_subplot(grid[0, -1])
    colorbar = figure.colorbar(mappable, cax=colorbar_axis)
    colorbar.set_ticks(colorbar_ticks)
    colorbar.set_ticklabels([f"{int(tick):,}" for tick in colorbar_ticks])
    colorbar.ax.minorticks_off()
    colorbar.set_label("Number of samples")

    # Align the colorbar to the plotting rectangle and keep it close to
    # the right-most Y axis. A linear scale is required to include zero.
    figure.canvas.draw()
    reference_box = axes[0].get_position()
    last_axis_box = axes[-1].get_position()
    colorbar_box = colorbar_axis.get_position()
    colorbar_axis.set_position([
        last_axis_box.x1 + 0.012,
        reference_box.y0,
        colorbar_box.width,
        reference_box.height,
    ])
    figure.suptitle(
        f"{forest} — GEDI RH95 versus canopy-height products\n"
        f"Our Model annual year-matched first; maximum valid TEST support per product; "
        f"canonical TEST n={canonical_n:,}",
        fontsize=15,
        fontweight="bold",
        y=0.965,
    )

    metrics_table = pd.DataFrame(rows)
    metrics_path = OUT_DIR / f"{forest}_all_CHM_scatter_one_row_metrics.csv"
    metrics_table.to_csv(metrics_path, index=False)
    print("Saved:", metrics_path)
    export_article_figure(figure, f"{forest}_all_CHM_scatter_one_row")
    plt.show()
    plt.close(figure)
    display(metrics_table)
    return metrics_table



# STEP_COMBINED_ECOSYSTEM_SCATTER_MATRIX_V5_COMPACT_METRICS
ECOSYSTEM_ROWS = [
    ("Ifran", "Moderately Dense\n(Ifran Forest)"),
    ("Maamoura", "Low Density\n(Maamoura Forest)"),
    ("Agadir", "Sparse\n(Agadir Forest)"),
]
# Left-to-right publication order: decreasing observed accuracy on the
# canonical/strict-common comparisons. Our Model remains the reference.
MATRIX_PRODUCTS = [
    "Our B4 Phase 2",
    "Pauls 2020",
    "Lang 2020",
    "Meta/Tolan 2023",
    "GFCH 2019",
]


def plot_combined_ecosystem_scatter_matrix(uniform_axis_45m=False, figure_stem_override=None, strict_common_support=False, latex_clean=False):
    def _format_metric(value, decimals=2, force_sign=False):
        # U+2212 is wider and clearer than the ASCII hyphen used by f-formatting.
        spec = f"+.{decimals}f" if force_sign else f".{decimals}f"
        return format(float(value), spec).replace("-", "\N{MINUS SIGN}")

    """One article figure: ecosystems in rows and CHM products in columns.

    When uniform_axis_45m=True, every panel uses 0–45 m and only the outer
    tick labels are displayed. The original ecosystem-specific version is
    retained for a direct visual publication comparison.
    """
    n_rows = len(ECOSYSTEM_ROWS)
    n_columns = len(MATRIX_PRODUCTS)

    # Optional strict support: every evaluable product in one ecosystem row
    # is assessed on exactly the same GEDI shot IDs. Products declared
    # non-evaluable are excluded from the intersection.
    strict_common_ids = {}
    if strict_common_support:
        for forest, _ in ECOSYSTEM_ROWS:
            cfg = SITES[forest]
            forest_samples = valid_samples[
                valid_samples["forest"].eq(forest)
            ]
            available_products = [
                product for product in MATRIX_PRODUCTS
                if product in cfg["products"]
            ]
            id_sets = []
            for product in available_products:
                ids = set(
                    forest_samples.loc[
                        forest_samples["product"].eq(product),
                        "shot_id",
                    ].astype(str)
                )
                if not ids:
                    raise AssertionError(
                        f"{forest}/{product}: empty GEDI support"
                    )
                id_sets.append(ids)
            common_ids = set.intersection(*id_sets)
            if len(common_ids) < 30:
                raise AssertionError(
                    f"{forest}: strict common GEDI support too small "
                    f"(n={len(common_ids)})"
                )
            strict_common_ids[forest] = frozenset(common_ids)
            print(
                f"[STRICT COMMON SUPPORT] {forest}: "
                f"n={len(common_ids):,} across {available_products}"
            )

    # Independent concentration scales per ecosystem row prevent the larger
    # Ifran sample from visually compressing Maamoura and Agadir.
    matrix_density = {}
    row_maximum_density = {
        forest: 0.0 for forest, _ in ECOSYSTEM_ROWS
    }
    for forest, _ in ECOSYSTEM_ROWS:
        cfg = SITES[forest]
        forest_samples = valid_samples[
            valid_samples["forest"].eq(forest)
        ]
        for product in MATRIX_PRODUCTS:
            if product not in cfg["products"]:
                continue
            part = forest_samples[
                forest_samples["product"].eq(product)
            ]
            if strict_common_support:
                part = part[
                    part["shot_id"].astype(str).isin(
                        strict_common_ids[forest]
                    )
                ]
            if part.empty:
                continue
            density = point_concentration_1m(part, float(cfg["eval_max"]))
            matrix_density[(forest, product)] = density
            row_maximum_density[forest] = max(
                row_maximum_density[forest],
                float(np.nanmax(density)),
            )

    row_colorbar_upper = {}
    row_colorbar_ticks = {}
    row_norm = {}
    reference_colorbar_scales = {
        "Ifran": (80.0, np.arange(0.0, 81.0, 20.0)),
        "Maamoura": (50.0, np.arange(0.0, 51.0, 10.0)),
        "Agadir": (4000.0, np.arange(0.0, 4001.0, 1000.0)),
    }
    for forest, _ in ECOSYSTEM_ROWS:
        if uniform_axis_45m and not latex_clean:
            # Preserve the published matrix graduations for direct visual
            # comparison with the original article figure.
            upper, ticks = reference_colorbar_scales[forest]
        else:
            upper, ticks = nice_colorbar_scale(
                row_maximum_density[forest]
            )
        row_colorbar_upper[forest] = upper
        row_colorbar_ticks[forest] = ticks
        # Linear normalization is mandatory here: equal numerical
        # differences must occupy equal physical distances on the colorbar.
        row_norm[forest] = (
            PowerNorm(gamma=0.38, vmin=0.0, vmax=upper)
            if latex_clean
            else Normalize(vmin=0.0, vmax=upper)
        )

    matrix_maximum_density = max(row_maximum_density.values())
    matrix_colorbar_upper, matrix_colorbar_ticks = nice_colorbar_scale(
        matrix_maximum_density
    )
    uniform_matrix_norm = PowerNorm(
        gamma=0.5,
        vmin=0.0,
        vmax=matrix_colorbar_upper,
    )

    figure, axes = plt.subplots(
        n_rows,
        n_columns,
        figsize=(
            (3.54 * n_columns, 11.0)
            if uniform_axis_45m
            else (19.0, 13.8)
        ),
        constrained_layout=False,
        squeeze=False,
        sharex=uniform_axis_45m,
        sharey=uniform_axis_45m,
    )
    if uniform_axis_45m:
        # Compact journal matrix: adjacent panels share their borders.
        if latex_clean:
            clean_left, clean_right = 0.018, 0.935
            clean_bottom, clean_top = 0.018, 0.988
            expected_ratio = (
                float(n_columns) * (clean_top - clean_bottom)
            ) / (3.0 * (clean_right - clean_left))
            figure.set_figheight(figure.get_figwidth() / expected_ratio)
            figure.subplots_adjust(
                left=clean_left, right=clean_right,
                bottom=clean_bottom, top=clean_top,
                wspace=0.0, hspace=0.0,
            )
        else:
            figure.subplots_adjust(
                left=0.060,
                right=0.925,
                bottom=0.075,
                top=0.91,
                wspace=0.0,
                hspace=0.0,
            )
        # With square 0–45 m panels, the figure aspect ratio must match
        # five panel widths by three panel heights. Otherwise Matplotlib
        # centers square axes inside over-tall cells and visually recreates
        # white row gaps despite hspace=0.
        expected_ratio = (
            (float(n_columns) * (0.988 - 0.018) / (3.0 * (0.935 - 0.018)))
            if latex_clean
            else (float(n_columns) * (0.91 - 0.075) / (3.0 * (0.925 - 0.060)))
        )
        actual_ratio = figure.get_figwidth() / figure.get_figheight()
        if not np.isclose(actual_ratio, expected_ratio, rtol=0.015):
            raise AssertionError(
                f"Uniform matrix geometry mismatch: {actual_ratio:.4f} "
                f"!= {expected_ratio:.4f}"
            )
    else:
        figure.subplots_adjust(
            left=0.065,
            right=0.94,
            bottom=0.07,
            top=0.91,
            wspace=0.10,
            hspace=0.16,
        )
    metric_rows = []

    for row_index, (forest, ecosystem_label) in enumerate(ECOSYSTEM_ROWS):
        cfg = SITES[forest]
        forest_samples = valid_samples[
            valid_samples["forest"].eq(forest)
        ].copy()
        native_axis_max = float(cfg["eval_max"])
        display_axis_max = 45.0 if uniform_axis_45m else native_axis_max

        for column_index, product in enumerate(MATRIX_PRODUCTS):
            axis = axes[row_index, column_index]
            axis.set_xlim(0, display_axis_max)
            axis.set_ylim(0, display_axis_max)
            axis.set_aspect("equal", adjustable="box")
            axis.set_box_aspect(1)
            axis.grid(True, color="0.86", linewidth=0.65, zorder=0)
            axis.plot(
                [0, display_axis_max],
                [0, display_axis_max],
                color="0.25",
                linestyle=":",
                linewidth=1.1,
                zorder=2,
            )
            axis.tick_params(labelsize=8.5)

            if uniform_axis_45m:
                axis.set_xlabel("")
                axis.set_ylabel("")
                axis.tick_params(
                    axis="x",
                    which="both",
                    bottom=(row_index == n_rows - 1),
                    labelbottom=(row_index == n_rows - 1),
                    top=False,
                    labeltop=False,
                )
                axis.tick_params(
                    axis="y",
                    which="both",
                    left=(column_index == 0),
                    labelleft=(column_index == 0),
                    right=False,
                    labelright=False,
                )
                # Avoid double-thick internal borders while keeping a single
                # continuous matrix grid, as in the reference article.
                # The internal vertical boundary belongs to the panel on
                # its right. Its x=0 tick and its y-axis spine are therefore
                # the exact same graphical coordinate, with no raster offset.
                axis.spines["left"].set_visible(True)
                axis.spines["left"].set_linewidth(0.85)
                axis.spines["left"].set_zorder(12)
                axis.spines["right"].set_visible(
                    column_index == n_columns - 1
                )
                if row_index < n_rows - 1:
                    axis.spines["bottom"].set_visible(False)
                if column_index == 0 and not latex_clean:
                    compact_row_label = {
                        "Ifran": "Moderately Dense (Ifran Forest)",
                        "Maamoura": "Low Density (Maamoura Forest)",
                        "Agadir": "Sparse (Agadir Forest)",
                    }[forest]
                    axis.text(
                        -0.13, 0.5, compact_row_label,
                        transform=axis.transAxes, rotation=90,
                        ha="center", va="center", fontsize=10.5,
                        fontweight="bold",
                    )
            else:
                if column_index == 0:
                    axis.set_ylabel(
                        ecosystem_label,
                        fontsize=10.5,
                        fontweight="bold",
                    )
                else:
                    axis.set_ylabel("")
                if row_index == n_rows - 1:
                    axis.set_xlabel("GEDI RH95 TEST (m)", fontsize=10)

            if row_index == 0 and not latex_clean:
                if uniform_axis_45m:
                    title = {
                        "Our B4 Phase 2": "Our Model",
                        "Lang 2020": "Lang et al. 2023 (L23)",
                        "Pauls 2020": "Pauls et al. 2024 (Pa24)",
                        "Meta/Tolan 2023": "Tolan et al. 2024 (T24)",
                        "GFCH 2019": "Potapov et al. 2021 (P21)",
                    }[product]
                else:
                    title = (
                        "Our Model\nannual year-matched"
                        if product == OUR_PRODUCT
                        else PRODUCT_LEGENDS[product]
                    )
                axis.set_title(title, fontsize=11.5, fontweight="bold", pad=8)

            if product not in cfg["products"]:
                # Keep the standard grid and x=y reference only. Do not draw a
                # large decorative X: it obscures the scientific reference line
                # and can extend into the adjacent colour bar after PDF export.
                axis.text(
                    0.5,
                    0.5,
                    "Not evaluable\n(insufficient valid coverage)",
                    transform=axis.transAxes,
                    ha="center",
                    va="center",
                    fontsize=11,
                    color="0.35",
                    bbox={
                        "facecolor": "white",
                        "edgecolor": "0.75",
                        "alpha": 0.92,
                    },
                )
                metric_rows.append({
                    "forest": forest,
                    "ecosystem": ecosystem_label,
                    "product": product,
                    "status": "NOT EVALUABLE",
                })
                continue

            part = forest_samples[
                forest_samples["product"].eq(product)
            ].copy()
            if strict_common_support:
                part = part[
                    part["shot_id"].astype(str).isin(
                        strict_common_ids[forest]
                    )
                ].copy()
                observed_ids = frozenset(part["shot_id"].astype(str))
                if observed_ids != strict_common_ids[forest]:
                    raise AssertionError(
                        f"{forest}/{product}: strict shot-ID mismatch "
                        f"({len(observed_ids)} != "
                        f"{len(strict_common_ids[forest])})"
                    )
            if part.empty:
                axis.text(
                    0.5, 0.5, "No valid TEST observations",
                    transform=axis.transAxes, ha="center", va="center",
                )
                metric_rows.append({
                    "forest": forest,
                    "ecosystem": ecosystem_label,
                    "product": product,
                    "status": "NO VALID TEST OBSERVATIONS",
                })
                continue

            # Preserve the established viridis concentration colors.
            # Low-density points are drawn first and dense cells last.
            density = matrix_density[(forest, product)]
            order = np.argsort(density, kind="mergesort")
            if uniform_axis_45m:
                # Publication-sized adaptive points with a common concentration
                # color scale. Dense cells are drawn last.
                point_size = float(
                    np.clip(34000.0 / max(len(part), 1), 5.0, 11.0)
                )
                axis.scatter(
                    part["rh95"].to_numpy(float)[order],
                    part["prediction"].to_numpy(float)[order],
                    c=density[order],
                    cmap="viridis",
                    norm=row_norm[forest],
                    s=point_size,
                    edgecolors="none",
                    alpha=0.88,
                    rasterized=True,
                    zorder=1,
                )
            else:
                axis.scatter(
                    part["rh95"].to_numpy(float)[order],
                    part["prediction"].to_numpy(float)[order],
                    c=density[order],
                    cmap="viridis",
                    norm=row_norm[forest],
                    s=8.0,
                    edgecolors="none",
                    alpha=0.78,
                    rasterized=True,
                    zorder=1,
                )

            metric = scatter_metrics(part)
            axis.text(
                0.025,
                0.975,
                (
                    # Compact publication summary on the exact support
                    # used by this panel.
                    f"R²={_format_metric(metric['r2'])}\n"
                    f"RMSE={_format_metric(metric['rmse'])} m\n"
                    f"MAE={_format_metric(metric['mae'])} m\n"
                    f"Bias={_format_metric(metric['bias'], force_sign=True)} m\n"
                    f"Std ratio={_format_metric(metric['std_ratio'])}\n"
                    f"n={metric['n']:,}"
                ),
                transform=axis.transAxes,
                ha="left",
                va="top",
                fontsize=8.4,
                fontfamily="DejaVu Sans",
                linespacing=1.12,
                bbox={
                    "facecolor": "white",
                    "edgecolor": "0.72",
                    "alpha": 0.70,
                    "pad": 2.2,
                },
                zorder=5,
            )
            metric_rows.append({
                "forest": forest,
                "ecosystem": ecosystem_label,
                "product": product,
                "status": "PASS",
                **metric,
            })

    # Equitable 5 m grid for the continuous journal-style matrix.
    # The former [0, 10, 20, 30, 40, 45] ticks produced four 10 m cells and
    # one 5 m cell. A constant 5 m interval now guarantees equally sized
    # squares in data space on both axes, including the terminal 40–45 m cell.
    # Do not call set_xticklabels/set_yticklabels on shared axes: their
    # Formatter is shared and would restore 45 on every internal boundary.
    # Tick-label visibility is changed on each individual Text artist instead.
    if uniform_axis_45m:
        equitable_grid_step_m = 5.0
        shared_ticks = np.arange(
            0.0,
            45.0 + 0.5 * equitable_grid_step_m,
            equitable_grid_step_m,
            dtype=float,
        )
        tick_intervals = np.diff(shared_ticks)
        if not np.allclose(tick_intervals, equitable_grid_step_m):
            raise AssertionError(
                f"Non-equitable scatter grid intervals: {tick_intervals}"
            )
        for row_index in range(n_rows):
            for column_index in range(n_columns):
                axis = axes[row_index, column_index]
                axis.set_xticks(shared_ticks)
                axis.set_yticks(shared_ticks)
                # A square plotting box plus identical X/Y limits makes every
                # 5 m by 5 m grid cell physically square.
                if not np.isclose(axis.get_data_ratio(), 1.0):
                    raise AssertionError(
                        f"Non-unit data ratio at row={row_index}, "
                        f"column={column_index}: {axis.get_data_ratio():.6f}"
                    )
                axis.tick_params(
                    axis="x",
                    bottom=(row_index == n_rows - 1),
                    labelbottom=(row_index == n_rows - 1),
                    top=False,
                    labeltop=False,
                )
                axis.tick_params(
                    axis="y",
                    left=(column_index == 0),
                    labelleft=(column_index == 0),
                    right=False,
                    labelright=False,
                )

        figure.canvas.draw()

        # X junctions: hide 45 from every preceding panel and retain the
        # geometrically coincident 0 from the next panel.
        for column_index in range(n_columns):
            axis = axes[-1, column_index]
            labels = axis.get_xticklabels()
            if len(labels) != len(shared_ticks):
                raise AssertionError(
                    f"Unexpected X tick-label count in column {column_index}: "
                    f"{len(labels)} != {len(shared_ticks)}"
                )
            if column_index < n_columns - 1:
                labels[-1].set_visible(False)
            labels[0].set_visible(True)
            labels[0].set_horizontalalignment("center")
            labels[0].set_x(0.0)

        # Y junctions: retain Y=0 of the upper panel and hide Y=45 from
        # every panel below it at the same horizontal boundary.
        for row_index in range(n_rows):
            axis = axes[row_index, 0]
            labels = axis.get_yticklabels()
            if len(labels) != len(shared_ticks):
                raise AssertionError(
                    f"Unexpected Y tick-label count in row {row_index}: "
                    f"{len(labels)} != {len(shared_ticks)}"
                )
            if row_index > 0:
                labels[-1].set_visible(False)
            labels[0].set_visible(True)

        # Geometry guard: all scatter plotting rectangles must touch.
        tolerance = 2e-4
        for row_index in range(n_rows):
            for column_index in range(n_columns - 1):
                left_box = axes[row_index, column_index].get_position()
                right_box = axes[row_index, column_index + 1].get_position()
                if abs(left_box.x1 - right_box.x0) > tolerance:
                    raise AssertionError(
                        f"Horizontal panel gap at row={row_index}, "
                        f"column={column_index}: {right_box.x0-left_box.x1:.6f}"
                    )
        for row_index in range(n_rows - 1):
            upper_box = axes[row_index, 0].get_position()
            lower_box = axes[row_index + 1, 0].get_position()
            if abs(upper_box.y0 - lower_box.y1) > tolerance:
                raise AssertionError(
                    f"Vertical panel gap at boundary={row_index}: "
                    f"{upper_box.y0-lower_box.y1:.6f}"
                )

        # Publication guard: inspect the actual major tick located at X=45.
        # get_xticklabels() omits invisible labels after a draw and must not
        # be used to identify the terminal tick.
        for column_index in range(n_columns - 1):
            terminal_ticks = [
                tick
                for tick in axes[-1, column_index].xaxis.get_major_ticks()
                if np.isclose(tick.get_loc(), 45.0)
            ]
            if len(terminal_ticks) != 1:
                raise AssertionError(
                    f"Expected one X=45 tick in column {column_index}, "
                    f"found {len(terminal_ticks)}"
                )
            if terminal_ticks[0].label1.get_visible():
                raise AssertionError(
                    f"Internal X=45 label still visible in column {column_index}"
                )

    # One independent concentration colorbar per ecosystem row.
    # Each bar uses that row's density range and exactly matches the plotting
    # rectangle in height; this preserves contrast across unequal sample sizes.
    figure.canvas.draw()
    for row_index, (forest, _) in enumerate(ECOSYSTEM_ROWS):
        last_box = axes[row_index, -1].get_position()
        # A small symmetric inset separates the bottom 0 tick from the
        # maximum tick of the colorbar immediately below, without changing
        # the zero-gap scatter matrix itself.
        colorbar_vertical_inset = 0.045 * last_box.height
        colorbar_axis = figure.add_axes([
            last_box.x1 + 0.010,
            last_box.y0 + colorbar_vertical_inset,
            0.012,
            last_box.height - 2.0 * colorbar_vertical_inset,
        ])
        scalar_mappable = plt.cm.ScalarMappable(
            norm=row_norm[forest],
            cmap="viridis",
        )
        scalar_mappable.set_array([])
        colorbar = figure.colorbar(
            scalar_mappable,
            cax=colorbar_axis,
        )
        ticks = np.asarray(row_colorbar_ticks[forest], dtype=float)
        if len(ticks) >= 3 and not np.allclose(
            np.diff(ticks), np.diff(ticks)[0]
        ):
            raise AssertionError(
                f"{forest}: colorbar ticks are not linearly spaced: {ticks}"
            )
        if len(ticks) == 0 or not np.isclose(ticks[0], 0.0):
            raise AssertionError(
                f"{forest}: colorbar must start at zero: {ticks}"
            )
        colorbar.set_ticks(ticks)
        colorbar.set_ticklabels([
            f"{int(tick):,}" for tick in ticks
        ])
        colorbar.ax.minorticks_off()
        colorbar.ax.tick_params(labelsize=8)

    if uniform_axis_45m:
        # Draw the three horizontal matrix borders in figure coordinates.
        # They cover panel-spine antialiasing at column junctions and produce
        # the uninterrupted journal-style grid used in the reference figure.
        matrix_left = axes[0, 0].get_position().x0
        matrix_right = axes[0, -1].get_position().x1
        horizontal_boundaries = [
            axes[0, 0].get_position().y1,
            *[
                axes[row_index, 0].get_position().y0
                for row_index in range(n_rows)
            ],
        ]
        for boundary_y in horizontal_boundaries:
            figure.add_artist(
                plt.Line2D(
                    [matrix_left, matrix_right],
                    [boundary_y, boundary_y],
                    transform=figure.transFigure,
                    color="0.10",
                    linewidth=0.85,
                    solid_capstyle="butt",
                    clip_on=False,
                    zorder=20,
                )
            )

    # Main caption is intentionally delegated to LaTeX.
    if uniform_axis_45m and not latex_clean:
        figure.supxlabel(
            "GEDI RH95 test (m)",
            x=0.50,
            y=0.041,
            fontsize=12,
        )
        figure.supylabel(
            "Canopy-height prediction (m)",
            x=0.012,
            y=0.50,
            fontsize=12,
        )
    if not uniform_axis_45m:
        figure.text(
            0.5,
            0.025,
            (
                "Primary protocol: all canonical TEST years; "
                "Our Model is exactly year-matched"
            ),
            ha="center",
            va="center",
            fontsize=10.5,
        )
    figure_stem = figure_stem_override or (
        "ALL_FORESTS_all_CHM_scatter_matrix_UNIFORM_45m"
        if uniform_axis_45m
        else "ALL_FORESTS_all_CHM_scatter_matrix"
    )
    export_article_figure(figure, figure_stem)
    plt.show()
    plt.close(figure)

    table = pd.DataFrame(metric_rows)
    table_path = OUT_DIR / (
        f"{figure_stem}_metrics.csv"
        if figure_stem_override
        else (
            "ALL_FORESTS_all_CHM_scatter_matrix_UNIFORM_45m_metrics.csv"
            if uniform_axis_45m
            else "ALL_FORESTS_all_CHM_scatter_matrix_metrics.csv"
        )
    )
    table.to_csv(table_path, index=False)
    print("Saved:", table_path)
    display(table)
    return table

def build_height_distribution_table(forest):
    cfg = SITES[forest]
    axis_max = float(cfg["eval_max"])
    edges = np.arange(0.0, axis_max + 1.0, 1.0)
    if not np.isclose(edges[-1], axis_max):
        edges = np.append(edges, axis_max)

    canonical = load_test_points(forest, cfg)
    series = [("GEDI TEST", canonical["rh95"].to_numpy(float))]
    forest_samples = valid_samples[valid_samples["forest"].eq(forest)]
    for product in cfg["products"]:
        heights = forest_samples.loc[
            forest_samples["product"].eq(product), "prediction"
        ].to_numpy(float)
        series.append((product, heights))

    rows = []
    for product, values in series:
        finite = values[np.isfinite(values)]
        in_domain = finite[(finite >= edges[0]) & (finite <= edges[-1])]
        if in_domain.size != finite.size:
            raise AssertionError(
                f"{forest}/{product}: {finite.size - in_domain.size} heights "
                f"outside plotting domain 0–{axis_max:g} m"
            )
        counts, _ = np.histogram(in_domain, bins=edges)
        percentages = 100.0 * counts / max(1, counts.sum())
        if counts.sum() and not np.isclose(percentages.sum(), 100.0):
            raise AssertionError((forest, product, percentages.sum()))
        for index, (count, percentage) in enumerate(zip(counts, percentages)):
            rows.append({
                "forest": forest,
                "product": product,
                "height_min_m": float(edges[index]),
                "height_max_m": float(edges[index + 1]),
                "count": int(count),
                "percent": float(percentage),
                "support_n": int(counts.sum()),
            })
    return pd.DataFrame(rows)


def plot_height_distributions(forest, distribution_table):
    cfg = SITES[forest]
    series_order = ["GEDI TEST"] + list(cfg["products"])
    n_rows = len(series_order)
    figure, axes = plt.subplots(
        n_rows,
        1,
        figsize=(10.0, 1.45 * n_rows + 1.7),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes)
    maximum_percentage = float(distribution_table["percent"].max())
    common_ymax = max(5.0, np.ceil(maximum_percentage / 5.0) * 5.0)

    for axis, product in zip(axes, series_order):
        part = distribution_table[distribution_table["product"].eq(product)]
        centers = 0.5 * (
            part["height_min_m"].to_numpy(float)
            + part["height_max_m"].to_numpy(float)
        )
        widths = (
            part["height_max_m"].to_numpy(float)
            - part["height_min_m"].to_numpy(float)
        )
        axis.bar(
            centers,
            part["percent"],
            width=0.92 * widths,
            color=ARTICLE_COLORS[product],
            edgecolor="white",
            linewidth=0.45,
        )
        support_n = int(part["support_n"].iloc[0])
        axis.text(
            0.012,
            0.82,
            product,
            transform=axis.transAxes,
            ha="left",
            va="top",
            fontsize=10.5,
            fontweight="bold" if product == "Our B4 Phase 2" else "normal",
            bbox={"facecolor": "white", "alpha": 0.76, "edgecolor": "none", "pad": 1.5},
        )
        axis.text(
            0.995,
            0.82,
            f"n={support_n:,}",
            transform=axis.transAxes,
            ha="right",
            va="top",
            fontsize=9,
        )
        axis.set_ylim(0, common_ymax)
        axis.set_ylabel("%")
        axis.grid(True, axis="y", alpha=0.18)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)

    axes[-1].set_xlim(0, cfg["eval_max"])
    axes[-1].set_xlabel("Canopy height (m)")
    figure.suptitle(
        f"{forest} — height distributions on canonical GEDI TEST support\n"
        "Our Model is year-matched; static CHMs use all canonical TEST years and their maximum valid support",
        fontsize=14,
        fontweight="bold",
    )
    export_article_figure(
        figure,
        f"{forest}_height_distributions_TEST_and_CHMs",
    )
    plt.show()
    plt.close(figure)


all_global_metrics = []
all_class_metrics = []

for forest_name in ("Ifran", "Maamoura", "Agadir"):
    print("\n" + "=" * 100)
    print(f"{forest_name} — article scatter and distribution figures")
    print("=" * 100)
    global_metrics = plot_ordered_scatter_row(forest_name)
    class_metrics = build_height_class_statistics(forest_name)
    class_metrics_path = OUT_DIR / f"{forest_name}_height_class_metrics.csv"
    class_metrics.to_csv(class_metrics_path, index=False)
    print("Saved:", class_metrics_path)
    plot_height_class_metric_table(forest_name, class_metrics)
    all_global_metrics.append(global_metrics)
    all_class_metrics.append(class_metrics)

combined_ecosystem_scatter_metrics = (
    plot_combined_ecosystem_scatter_matrix(
        uniform_axis_45m=False
    )
)
combined_uniform_45m_scatter_metrics = (
    plot_combined_ecosystem_scatter_matrix(
        uniform_axis_45m=True
    )
)

article_global_metrics = pd.concat(all_global_metrics, ignore_index=True)
article_height_class_metrics = pd.concat(all_class_metrics, ignore_index=True)
article_global_metrics.to_csv(
    OUT_DIR / "ALL_FORESTS_scatter_global_metrics.csv", index=False
)
article_height_class_metrics.to_csv(
    OUT_DIR / "ALL_FORESTS_height_class_metrics.csv", index=False
)
print("\n[PASS] Article figures complete.")
print("[PASS] Our Model annual year-matched is the left-most scatter panel for every forest.")
print("[PASS] No signed-error or height-distribution plot is generated.")
print("[PASS] RMSE^2 = Bias^2 + SDSD + LCS was checked wherever defined.")


# Chapter 5 — Strict year-matched temporal sensitivity

This supplementary analysis quantifies the effect of temporal mismatch without changing the primary TEST benchmark.

For each static product having a defensible nominal year:

- the canonical TEST split is filtered to that year only;
- Our Model uses its annual map for the same year;
- both products are restricted to the exact same valid `shot_id` intersection;
- no product is selected or tuned from these TEST results.

The pre-declared pairs are:

1. **Our Model 2019 vs Potapov/GFCH 2019**, evaluated on GEDI TEST 2019;
2. **Our Model 2020 vs Lang 2020**, evaluated on GEDI TEST 2020.

Meta/Tolan is excluded from strict-year matching because its imagery has varying acquisition dates and does not represent a unique 2023 snapshot. Missing forest/product coverage and years with no canonical TEST shots are reported as `NOT EVALUABLE`, never silently discarded.


In [ ]:
# STEP_TEMPORAL_SENSITIVITY_STRICT_YEAR_V1
STRICT_YEAR_PRODUCTS = {
    "GFCH 2019": 2019,
    "Lang 2020": 2020,
    "Pauls 2020": 2020,
}


def paired_strict_year_frames(forest, competitor, year):
    cfg = SITES[forest]
    canonical = load_test_points(forest, cfg)
    canonical_year = canonical[
        canonical["gedi_year"].astype(int).eq(int(year))
    ].copy()
    if canonical_year.empty:
        return None, {
            "forest": forest,
            "competitor": competitor,
            "nominal_year": int(year),
            "canonical_year_test_n": 0,
            "paired_valid_n": 0,
            "status": f"NOT EVALUABLE: no canonical TEST shot in {year}",
        }
    if competitor not in cfg["products"]:
        return None, {
            "forest": forest,
            "competitor": competitor,
            "nominal_year": int(year),
            "canonical_year_test_n": int(canonical_year["shot_id"].nunique()),
            "paired_valid_n": 0,
            "status": "NOT EVALUABLE: product unavailable for this forest",
        }

    forest_samples = valid_samples[
        valid_samples["forest"].eq(forest)
        & valid_samples["gedi_year"].astype(int).eq(int(year))
    ].copy()
    ours = forest_samples[forest_samples["product"].eq(OUR_PRODUCT)].copy()
    other = forest_samples[forest_samples["product"].eq(competitor)].copy()
    paired_ids = set(ours["shot_id"].astype(str)).intersection(
        set(other["shot_id"].astype(str))
    )
    if not paired_ids:
        return None, {
            "forest": forest,
            "competitor": competitor,
            "nominal_year": int(year),
            "canonical_year_test_n": int(canonical_year["shot_id"].nunique()),
            "paired_valid_n": 0,
            "status": "NOT EVALUABLE: no common valid raster support",
        }

    ours = ours[ours["shot_id"].astype(str).isin(paired_ids)].copy()
    other = other[other["shot_id"].astype(str).isin(paired_ids)].copy()
    if frozenset(ours["shot_id"].astype(str)) != frozenset(
        other["shot_id"].astype(str)
    ):
        raise AssertionError(f"{forest}/{competitor}/{year}: shot_id mismatch")
    if not (
        ours["gedi_year"].astype(int).eq(int(year))
        & ours["product_year"].astype(int).eq(int(year))
    ).all():
        raise AssertionError(f"{forest}/Our Model: strict-year guard failed")

    pair = pd.concat([ours, other], ignore_index=True)
    pair["strict_pair"] = f"Our Model {year} vs {competitor}"
    return pair, {
        "forest": forest,
        "competitor": competitor,
        "nominal_year": int(year),
        "canonical_year_test_n": int(canonical_year["shot_id"].nunique()),
        "paired_valid_n": int(len(paired_ids)),
        "status": "PASS",
    }


def plot_strict_year_pair(forest, competitor, year, pair):
    cfg = SITES[forest]
    products = [OUR_PRODUCT, competitor]
    density = {
        product: point_concentration_1m(
            pair[pair["product"].eq(product)], cfg["eval_max"]
        )
        for product in products
    }
    maximum_density = max(float(np.max(values)) for values in density.values())
    colorbar_upper, colorbar_ticks = nice_colorbar_scale(maximum_density)
    shared_norm = Normalize(vmin=0.0, vmax=colorbar_upper)

    figure = plt.figure(figsize=(11.8, 6.3), constrained_layout=False)
    grid = figure.add_gridspec(
        1, 3,
        width_ratios=[1.0, 1.0, 0.045],
        left=0.07, right=0.95, bottom=0.12, top=0.80, wspace=0.18,
    )
    axes = [figure.add_subplot(grid[0, 0])]
    axes.append(figure.add_subplot(grid[0, 1], sharex=axes[0], sharey=axes[0]))
    mappable = None
    metric_rows = []

    for index, (axis, product) in enumerate(zip(axes, products)):
        part = pair[pair["product"].eq(product)].copy()
        order = np.argsort(density[product], kind="mergesort")
        mappable = axis.scatter(
            part["rh95"].to_numpy(float)[order],
            part["prediction"].to_numpy(float)[order],
            c=density[product][order],
            cmap="viridis", norm=shared_norm,
            s=16, edgecolors="none", alpha=0.78,
            rasterized=True,
        )
        axis.plot(
            [0, cfg["eval_max"]], [0, cfg["eval_max"]],
            color="0.25", linestyle="--", linewidth=1.0,
        )
        axis.set_xlim(0, cfg["eval_max"])
        axis.set_ylim(0, cfg["eval_max"])
        axis.set_aspect("equal", adjustable="box")
        axis.set_box_aspect(1)
        axis.set_xlabel(f"GEDI RH95 TEST {year} (m)")
        axis.set_ylabel("CHM height (m)")
        title = (
            f"Our Model map {year}"
            if product == OUR_PRODUCT
            else PRODUCT_LEGENDS[product]
        )
        axis.set_title(title)
        metric = extended_schwartz_metrics(part)
        basic_metric = scatter_metrics(part)
        metric_rows.append({
            "forest": forest,
            "product": product,
            "comparison_product": competitor,
            "strict_year": int(year),
            "support_mode": "exact same GEDI TEST shot_id intersection",
            **metric,
            "mae_m": basic_metric["mae"],
            "r2": basic_metric["r2"],
            "slope": basic_metric["slope"],
        })
        axis.text(
            0.025, 0.975,
            (
                f"R²={basic_metric['r2']:.2f}\n"
                f"RMSE={basic_metric['rmse']:.2f} m\n"
                f"MAE={basic_metric['mae']:.2f} m\n"
                f"Bias={basic_metric['bias']:+.2f} m\n"
                f"Slope={basic_metric['slope']:.2f}\n"
                f"Corr={basic_metric['corr']:.2f}\n"
                f"n={basic_metric['n']:,}"
            ),
            transform=axis.transAxes,
            ha="left", va="top", fontsize=9,
            bbox={"facecolor": "white", "alpha": 0.88, "edgecolor": "0.75"},
            zorder=5,
        )
        axis.text(
            0.01, 1.035, f"({chr(97 + index)})",
            transform=axis.transAxes, fontweight="bold",
        )

    colorbar_axis = figure.add_subplot(grid[0, 2])
    colorbar = figure.colorbar(mappable, cax=colorbar_axis)
    colorbar.set_ticks(colorbar_ticks)
    colorbar.set_ticklabels([f"{int(tick):,}" for tick in colorbar_ticks])
    colorbar.ax.minorticks_off()
    colorbar.set_label("Number of samples")
    figure.canvas.draw()
    reference_box = axes[0].get_position()
    last_axis_box = axes[-1].get_position()
    colorbar_box = colorbar_axis.get_position()
    colorbar_axis.set_position([
        last_axis_box.x1 + 0.012,
        reference_box.y0,
        colorbar_box.width,
        reference_box.height,
    ])
    figure.suptitle(
        f"{forest} — strict temporal sensitivity for {year}\n"
        f"Exact paired canonical GEDI TEST support: n={pair['shot_id'].nunique():,}",
        fontsize=14, fontweight="bold", y=0.96,
    )
    safe = competitor.replace(" ", "_").replace("/", "_")
    export_article_figure(
        figure, f"{forest}_STRICT_YEAR_{year}_Our_Model_vs_{safe}"
    )
    plt.show()
    plt.close(figure)
    return metric_rows


strict_support_rows = []
strict_metric_rows = []
strict_year_count_rows = []

for forest, cfg in SITES.items():
    canonical = load_test_points(forest, cfg)
    for year, count in (
        canonical.groupby("gedi_year")["shot_id"].nunique().sort_index().items()
    ):
        strict_year_count_rows.append({
            "forest": forest,
            "gedi_year": int(year),
            "canonical_test_n": int(count),
        })

    for competitor, year in STRICT_YEAR_PRODUCTS.items():
        pair, audit = paired_strict_year_frames(
            forest, competitor, year
        )
        strict_support_rows.append(audit)
        print(
            forest, "|", competitor, "|", year, "|",
            audit["status"], "| paired n =", audit["paired_valid_n"],
        )
        if pair is not None:
            strict_metric_rows.extend(
                plot_strict_year_pair(forest, competitor, year, pair)
            )

    strict_support_rows.append({
        "forest": forest,
        "competitor": "Meta/Tolan 2023",
        "nominal_year": np.nan,
        "canonical_year_test_n": np.nan,
        "paired_valid_n": np.nan,
        "status": (
            "NOT APPLICABLE: varying imagery acquisition dates; "
            "no artificial 2023 match"
        ),
    })

strict_year_counts = pd.DataFrame(strict_year_count_rows)
strict_temporal_support = pd.DataFrame(strict_support_rows)
strict_temporal_metrics = pd.DataFrame(strict_metric_rows)

strict_year_counts.to_csv(
    OUT_DIR / "STRICT_YEAR_canonical_TEST_counts.csv", index=False
)
strict_temporal_support.to_csv(
    OUT_DIR / "STRICT_YEAR_support_audit.csv", index=False
)
strict_temporal_metrics.to_csv(
    OUT_DIR / "STRICT_YEAR_paired_metrics.csv", index=False
)

display(strict_year_counts)
display(strict_temporal_support)
display(strict_temporal_metrics)
print("[PASS] Protocol B uses exact years and exact paired shot_id support.")
print("[PASS] Meta/Tolan was not assigned an artificial 2023 map year.")


## Article figures — Pauls baseline, height distributions and binned errors

This final chapter follows the visual logic used in global canopy-height benchmarks while preserving the study's canonical TEST protocol.

- **Scatter panels:** each product uses its maximum valid canonical TEST support; Our Model remains exactly year-matched to each GEDI acquisition.
- **Height distributions:** GEDI labels and CHM predictions are displayed as aligned 1 m histograms on a logarithmic frequency scale.
- **Binned-error boxplots:** products are compared only on the strict intersection of valid GEDI shots, using 0–5, 5–10 m, etc. classes. The signed error is CHM minus GEDI RH95.
- Pauls et al. (2024) is the 2020 global 10 m product (`Pa24`).

All figures are exported as PNG, SVG and PDF.


In [ ]:
# STEP_ARTICLE_ECHOSAT_PAULS_COMPOSITES_V1
from matplotlib.gridspec import GridSpec

ARTICLE_PRODUCT_ORDER = [
    OUR_PRODUCT,
    "Pauls 2020",
    "Lang 2020",
    "Meta/Tolan 2023",
    "GFCH 2019",
]
ARTICLE_SHORT_LABELS = {
    OUR_PRODUCT: "Our Model",
    "Lang 2020": "Lang et al. (L23)",
    "Meta/Tolan 2023": "Tolan et al. (T24)",
    "Pauls 2020": "Pauls et al. (Pa24)",
    "GFCH 2019": "Potapov et al. (P21)",
}


def _forest_product_frame(forest, product):
    return valid_samples.loc[
        valid_samples["forest"].eq(forest)
        & valid_samples["product"].eq(product)
        & np.isfinite(valid_samples["rh95"])
        & np.isfinite(valid_samples["prediction"])
    ].copy()


def plot_echosat_style_benchmark(forest):
    cfg = SITES[forest]
    products = [p for p in ARTICLE_PRODUCT_ORDER if p in cfg["products"]]
    axis_max = float(cfg["eval_max"])
    forest_all = valid_samples.loc[valid_samples["forest"].eq(forest)].copy()
    gedi = (
        forest_all.sort_values("shot_id")
        .drop_duplicates("shot_id")["rh95"]
        .dropna().to_numpy(float)
    )

    figure = plt.figure(figsize=(19.0, 8.2), constrained_layout=True)
    outer = figure.add_gridspec(1, 2, width_ratios=[1.62, 1.0], wspace=0.10)
    scatter_grid = outer[0].subgridspec(2, 3, wspace=0.16, hspace=0.20)
    hist_grid = outer[1].subgridspec(len(products) + 1, 1, hspace=0.08)

    scatter_axes = [figure.add_subplot(scatter_grid[i // 3, i % 3]) for i in range(6)]
    for index, product in enumerate(products):
        axis = scatter_axes[index]
        part = _forest_product_frame(forest, product)
        metric = scatter_metrics(part)
        axis.hexbin(
            part["rh95"], part["prediction"],
            gridsize=42, extent=(0, axis_max, 0, axis_max),
            mincnt=1, bins="log", cmap="viridis",
            linewidths=0, rasterized=True,
        )
        axis.plot([0, axis_max], [0, axis_max], "k--", lw=1.0)
        axis.set_xlim(0, axis_max)
        axis.set_ylim(0, axis_max)
        axis.set_aspect("equal", adjustable="box")
        axis.set_title(ARTICLE_SHORT_LABELS[product], fontsize=11.5)
        axis.grid(True, alpha=0.16, linewidth=0.6)
        if index // 3 == 1:
            axis.set_xlabel("GEDI RH95 height class (m)")
        if index % 3 == 0:
            axis.set_ylabel("Signed error (CHM \u2212 GEDI RH95, m)")
        axis.text(
            0.04, 0.96,
            f"n={int(metric['n']):,}\n"
            f"R²={metric['r2']:.2f}\n"
            f"RMSE={metric['rmse']:.2f} m\n"
            f"MAE={metric['mae']:.2f} m\n"
            f"Bias={metric['bias']:+.2f} m",
            transform=axis.transAxes, ha="left", va="top", fontsize=8.4,
            bbox={"facecolor": "white", "alpha": 0.84, "edgecolor": "0.65", "pad": 2.0},
        )
    for axis in scatter_axes[len(products):]:
        axis.set_axis_off()

    distributions = [("GEDI TEST", gedi, "#D8D8D8")]
    distributions.extend(
        (
            ARTICLE_SHORT_LABELS[product],
            _forest_product_frame(forest, product)["prediction"].to_numpy(float),
            COLORS[product],
        )
        for product in products
    )
    bins = np.arange(0.0, axis_max + 1.0001, 1.0)
    for index, (label, values, color) in enumerate(distributions):
        axis = figure.add_subplot(hist_grid[index, 0])
        values = values[np.isfinite(values) & (values >= 0) & (values <= axis_max)]
        axis.hist(values, bins=bins, color=color, alpha=0.88, edgecolor="white", linewidth=0.25)
        axis.set_yscale("log")
        axis.set_ylim(bottom=0.8)
        axis.set_xlim(0, axis_max)
        axis.grid(True, axis="x", alpha=0.18, linewidth=0.6)
        axis.text(0.98, 0.78, label, transform=axis.transAxes, ha="right", va="top", fontsize=10.5)
        axis.text(0.98, 0.18, f"n={len(values):,}", transform=axis.transAxes, ha="right", va="bottom", fontsize=8)
        if index < len(distributions) - 1:
            axis.tick_params(labelbottom=False)
        else:
            axis.set_xlabel("GEDI RH95 height class (m)")
        if index == len(distributions) // 2:
            axis.set_ylabel("Signed error (CHM \u2212 GEDI RH95, m)")

    # Figure caption is supplied by LaTeX.
    export_article_figure(figure, f"{forest}_ECHOSAT_style_scatter_and_height_distributions")
    plt.show()
    plt.close(figure)


def plot_binned_error_boxplots(forest):
    cfg = SITES[forest]
    products = [p for p in ARTICLE_PRODUCT_ORDER if p in cfg["products"]]
    forest_all = valid_samples.loc[valid_samples["forest"].eq(forest)].copy()
    id_sets = {
        product: set(
            forest_all.loc[forest_all["product"].eq(product), "shot_id"].astype(str)
        )
        for product in products
    }
    common_ids = set.intersection(*(ids for ids in id_sets.values()))
    if len(common_ids) < 30:
        raise RuntimeError(f"{forest}: strict all-product support is too small (n={len(common_ids)}).")

    edges = np.asarray(cfg["height_edges"], dtype=float)
    labels = [f"{edges[i]:g}–{edges[i + 1]:g} m" for i in range(len(edges) - 1)]
    centers = np.arange(len(labels), dtype=float)
    total_width = 0.84
    box_width = total_width / len(products)
    figure, axis = plt.subplots(figsize=(11.8, 4.25))
    audit_rows = []

    for product_index, product in enumerate(products):
        part = forest_all.loc[
            forest_all["product"].eq(product)
            & forest_all["shot_id"].astype(str).isin(common_ids)
        ].copy()
        part["height_bin"] = pd.cut(
            part["rh95"], bins=edges, labels=labels,
            include_lowest=True, right=True,
        )
        part["signed_error"] = part["prediction"] - part["rh95"]
        series = [
            part.loc[part["height_bin"].astype(str).eq(label), "signed_error"]
            .dropna().to_numpy(float)
            for label in labels
        ]
        positions = centers - total_width / 2 + box_width / 2 + product_index * box_width
        bp = axis.boxplot(
            series, positions=positions, widths=box_width * 0.90,
            patch_artist=True, showfliers=False, whis=1.5,
            medianprops={"color": "black", "linewidth": 1.0},
            whiskerprops={"color": "0.42", "linewidth": 0.8},
            capprops={"color": "0.42", "linewidth": 0.8},
            manage_ticks=False,
        )
        for patch in bp["boxes"]:
            patch.set_facecolor(COLORS[product])
            patch.set_edgecolor("0.35")
        for label in labels:
            values = part.loc[part["height_bin"].astype(str).eq(label), "signed_error"].dropna()
            audit_rows.append({
                "forest": forest, "product": product, "height_class": label,
                "strict_common_n_total": len(common_ids), "class_n": int(len(values)),
                "mean_error_m": float(values.mean()) if len(values) else np.nan,
                "median_error_m": float(values.median()) if len(values) else np.nan,
            })

    axis.axhline(0, color="black", linestyle="--", linewidth=1.0)
    axis.set_xticks(centers)
    axis.set_xticklabels(labels, rotation=18, ha="right")
    # Axis prose is composed as native LaTeX in the article.
    axis.set_xlabel("")
    axis.set_ylabel("")
    # Forest and support n belong in the LaTeX caption.
    axis.grid(True, axis="y", alpha=0.20)
    # Product-colour mapping is supplied by the LaTeX caption.
    figure.subplots_adjust(left=0.055, right=0.995, bottom=0.16, top=0.985)
    export_article_figure(figure, f"{forest}_binned_error_boxplots_strict_common_TEST")
    plt.show()
    plt.close(figure)

    audit = pd.DataFrame(audit_rows)
    colour_key = pd.DataFrame([
        {
            "forest": forest,
            "product": product,
            "label": ARTICLE_SHORT_LABELS[product],
            "matplotlib_color": COLORS[product],
        }
        for product in products
    ])
    colour_key_path = OUT_DIR / f"{forest}_binned_error_colour_key.csv"
    colour_key.to_csv(colour_key_path, index=False)
    print("Saved:", colour_key_path)
    audit_path = OUT_DIR / f"{forest}_binned_error_boxplots_strict_common_TEST.csv"
    audit.to_csv(audit_path, index=False)
    print("Saved:", audit_path)
    return audit


### Standalone height distributions

Height distributions are exported as independent figures. They are deliberately
separated from the scatter plots so that each forest-level distribution figure
can be positioned and captioned independently in LaTeX.

Each figure contains the canonical GEDI TEST distribution followed by Our Model
and every evaluable global CHM. Frequencies use a logarithmic scale and identical
one-metre height classes within a forest.


In [ ]:
# STEP_RUN_HEIGHT_DISTRIBUTIONS_ONLY_V2
HEIGHT_DISTRIBUTION_ORDER = [
    OUR_PRODUCT,
    "Lang 2020",
    "Pauls 2020",
    "Meta/Tolan 2023",
    "GFCH 2019",
]


def plot_standalone_height_distributions(forest):
    cfg = SITES[forest]
    axis_max = float(cfg["eval_max"])
    products = [
        product
        for product in HEIGHT_DISTRIBUTION_ORDER
        if product in cfg["products"]
    ]

    canonical = load_test_points(forest, cfg)
    distributions = [
        (
            "GEDI TEST",
            canonical["rh95"].to_numpy(float),
            "#D9D9D9",
        )
    ]
    for product in products:
        frame = _forest_product_frame(forest, product)
        distributions.append(
            (
                ARTICLE_SHORT_LABELS[product],
                frame["prediction"].to_numpy(float),
                COLORS[product],
            )
        )

    bins = np.arange(0.0, axis_max + 1.0001, 1.0)
    figure, axes = plt.subplots(
        len(distributions), 1,
        figsize=(9.2, 1.30 * len(distributions) + 0.85),
        sharex=True,
        constrained_layout=False,
    )
    axes = np.atleast_1d(axes)
    figure.subplots_adjust(
        left=0.105, right=0.985, bottom=0.105, top=0.985,
        hspace=0.10,
    )
    audit_rows = []

    for index, (label, values, color) in enumerate(distributions):
        axis = axes[index]
        values = np.asarray(values, dtype=float)
        values = values[
            np.isfinite(values)
            & (values >= 0.0)
            & (values <= axis_max)
        ]
        counts, _ = np.histogram(values, bins=bins)
        axis.bar(
            bins[:-1], counts,
            width=np.diff(bins),
            align="edge",
            color=color,
            edgecolor="white",
            linewidth=0.35,
            alpha=0.90,
        )
        axis.set_yscale("log")
        axis.set_ylim(bottom=0.8)
        axis.set_xlim(0.0, axis_max)
        axis.grid(True, axis="x", color="0.88", linewidth=0.55)
        axis.text(
            0.985, 0.78, label,
            transform=axis.transAxes,
            ha="right", va="top",
            fontsize=10.2,
            fontweight="bold" if label == "Our Model" else "normal",
        )
        axis.text(
            0.985, 0.14, f"n={len(values):,}",
            transform=axis.transAxes,
            ha="right", va="bottom", fontsize=8.3,
        )
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
        if index < len(distributions) - 1:
            axis.tick_params(labelbottom=False)
        else:
            axis.set_xlabel("Canopy height (m)")
        if index == len(distributions) // 2:
            axis.set_ylabel("Frequency (log scale)")

        for bin_index, count in enumerate(counts):
            audit_rows.append({
                "forest": forest,
                "series": label,
                "height_min_m": float(bins[bin_index]),
                "height_max_m": float(bins[bin_index + 1]),
                "count": int(count),
                "support_n": int(len(values)),
            })

    stem = f"{forest}_standalone_height_distributions"
    export_article_figure(figure, stem)
    audit = pd.DataFrame(audit_rows)
    audit_path = OUT_DIR / f"{stem}.csv"
    audit.to_csv(audit_path, index=False)
    print("Saved:", audit_path)
    plt.show()
    plt.close(figure)
    return audit


standalone_height_distribution_audits = []
for forest in SITES:
    print(f"\n[STANDALONE HEIGHT DISTRIBUTIONS] {forest}")
    standalone_height_distribution_audits.append(
        plot_standalone_height_distributions(forest)
    )

standalone_height_distribution_audit = pd.concat(
    standalone_height_distribution_audits,
    ignore_index=True,
)
display(standalone_height_distribution_audit)
print("[PASS] Height distributions exported separately as PNG, SVG and PDF.")


### Binned error distributions for the three forests

These are the grouped boxplots requested for the article. Each forest gets a separate figure. Errors are evaluated on the strict common canonical TEST support of all available products, including Pauls.


In [ ]:
# STEP_RUN_BINNED_ERROR_BOXPLOTS_THREE_FORESTS_V1
article_binned_error_tables = []
for forest in SITES:
    print(f"\n[BINNED ERROR BOXPLOT] {forest}")
    article_binned_error_tables.append(plot_binned_error_boxplots(forest))

article_binned_error_summary = pd.concat(
    article_binned_error_tables, ignore_index=True
)
display(article_binned_error_summary)
print("[PASS] Three independent binned-error boxplots generated.")
print("[PASS] Classes begin at 0–5 m, followed by 5–10 m, etc.")
print("[PASS] PNG, SVG, PDF and CSV audit files saved in:", OUT_DIR)


### Asymmetric article scatter layout

For each forest, **Our Model** is isolated in a large panel on the left. The four
comparison CHMs form a contiguous 2 × 2 grid on the right. The comparison panels
share exactly the same X and Y scales and have zero horizontal and vertical
spacing. The global scientific title is intentionally left to the LaTeX caption.


In [ ]:
# STEP_ARTICLE_ASYMMETRIC_SCATTER_LAYOUT_V3_ZERO_GAP
from matplotlib.colors import PowerNorm
from matplotlib.gridspec import GridSpec

ASYMMETRIC_PRODUCT_ORDER = [
    "Lang 2020",
    "Pauls 2020",
    "Meta/Tolan 2023",
    "GFCH 2019",
]

ASYMMETRIC_LABELS = {
    OUR_PRODUCT: "Our Model",
    "Lang 2020": "Lang et al. (L23)",
    "Pauls 2020": "Pauls et al. (Pa24)",
    "Meta/Tolan 2023": "Tolan et al. (T24)",
    "GFCH 2019": "Potapov et al. (P21)",
}


def _article_ticks(axis_max):
    if np.isclose(axis_max, 20.0):
        return np.asarray([0, 5, 10, 15, 20], dtype=float)
    if np.isclose(axis_max, 45.0):
        return np.asarray([0, 10, 20, 30, 40, 45], dtype=float)
    step = 5.0 if axis_max <= 25 else 10.0
    ticks = np.arange(0.0, axis_max, step)
    return np.append(ticks, axis_max)


def _asymmetric_metric_row(forest, product, frame):
    metric = scatter_metrics(frame)
    return {
        "forest": forest,
        "product": product,
        "label": ASYMMETRIC_LABELS[product],
        "status": "PASS",
        "n": int(metric["n"]),
        "r2": float(metric["r2"]),
        "corr": float(metric["corr"]),
        "rmse_m": float(metric["rmse"]),
        "mae_m": float(metric["mae"]),
        "bias_m": float(metric["bias"]),
    }


def _draw_asymmetric_scatter(
    axis, forest, product, frame, density, norm, axis_max, letter
):
    order = np.argsort(density, kind="mergesort")
    mappable = axis.scatter(
        frame["rh95"].to_numpy(float)[order],
        frame["prediction"].to_numpy(float)[order],
        c=density[order],
        cmap="viridis",
        norm=norm,
        s=float(np.clip(28000.0 / max(len(frame), 1), 5.0, 14.0)),
        edgecolors="none",
        alpha=0.90,
        rasterized=True,
        zorder=2,
    )
    axis.plot(
        [0, axis_max], [0, axis_max],
        color="0.18", linestyle="--", linewidth=0.9, zorder=3,
    )
    axis.set_title(ASYMMETRIC_LABELS[product], fontsize=11.2, pad=5)
    metric = scatter_metrics(frame)
    axis.text(
        0.025, 0.975,
        f"R²={metric['r2']:.2f}\n"
        f"Corr={metric['corr']:.2f}\n"
        f"RMSE={metric['rmse']:.2f} m\n"
        f"MAE={metric['mae']:.2f} m\n"
        f"Bias={metric['bias']:+.2f} m\n"
        f"n={metric['n']:,}",
        transform=axis.transAxes,
        ha="left", va="top",
        fontsize=8.2, linespacing=1.05,
        bbox={
            "facecolor": "white", "edgecolor": "0.68",
            "alpha": 0.90, "pad": 2.0,
        },
        zorder=5,
    )
    axis.text(
        0.985, 0.02, f"({letter})",
        transform=axis.transAxes,
        ha="right", va="bottom",
        fontsize=9.5, fontweight="bold", zorder=6,
    )
    return mappable


def _draw_asymmetric_unavailable(axis, product, axis_max, letter):
    axis.plot(
        [0, axis_max], [0, axis_max],
        color="0.55", linestyle="--", linewidth=0.9,
    )
    axis.set_title(ASYMMETRIC_LABELS[product], fontsize=11.2, pad=5)
    axis.text(
        0.5, 0.5,
        "Not evaluable\n(insufficient valid coverage)",
        transform=axis.transAxes,
        ha="center", va="center",
        fontsize=9.5, color="0.35",
        bbox={"facecolor": "white", "edgecolor": "0.72", "alpha": 0.92},
    )
    axis.text(
        0.985, 0.02, f"({letter})",
        transform=axis.transAxes,
        ha="right", va="bottom",
        fontsize=9.5, fontweight="bold",
    )


def plot_asymmetric_article_scatter_v3(forest):
    cfg = SITES[forest]
    axis_max = float(cfg["eval_max"])
    ticks = _article_ticks(axis_max)
    forest_samples = valid_samples[
        valid_samples["forest"].eq(forest)
    ].copy()
    products_for_density = [
        OUR_PRODUCT,
        *[
            product
            for product in ASYMMETRIC_PRODUCT_ORDER
            if product in cfg["products"]
        ],
    ]
    frames = {
        product: _forest_product_frame(forest, product)
        for product in products_for_density
    }
    densities = {
        product: point_concentration_1m(frame, axis_max)
        for product, frame in frames.items()
    }
    maximum_density = max(
        float(np.nanmax(values)) for values in densities.values()
    )
    colorbar_upper, colorbar_ticks = nice_colorbar_scale(maximum_density)
    shared_norm = PowerNorm(
        gamma=0.50, vmin=0.0, vmax=colorbar_upper,
    )

    # The plotting rectangle follows 3.20 width units × 2 height units.
    # This makes every comparison panel square while hspace=wspace=0.
    figure = plt.figure(figsize=(16.06, 9.8), constrained_layout=False)
    grid = GridSpec(
        2, 4, figure=figure,
        width_ratios=[1.0, 0.20, 1.0, 1.0],
        left=0.065, right=0.90, bottom=0.085, top=0.94,
        wspace=0.0, hspace=0.0,
    )

    our_axis = figure.add_subplot(grid[:, 0])
    comparison_axes = {
        "Lang 2020": figure.add_subplot(
            grid[0, 2], sharex=our_axis, sharey=our_axis
        ),
        "Pauls 2020": figure.add_subplot(
            grid[0, 3], sharex=our_axis, sharey=our_axis
        ),
        "Meta/Tolan 2023": figure.add_subplot(
            grid[1, 2], sharex=our_axis, sharey=our_axis
        ),
        "GFCH 2019": figure.add_subplot(
            grid[1, 3], sharex=our_axis, sharey=our_axis
        ),
    }
    axes = [our_axis, *comparison_axes.values()]
    for axis in axes:
        axis.set_xlim(0.0, axis_max)
        axis.set_ylim(0.0, axis_max)
        axis.set_xticks(ticks)
        axis.set_yticks(ticks)
        axis.set_aspect("equal", adjustable="box")
        axis.set_box_aspect(1)
        axis.grid(True, color="0.87", linewidth=0.55, zorder=0)

    rows = []
    mappable = _draw_asymmetric_scatter(
        our_axis, forest, OUR_PRODUCT,
        frames[OUR_PRODUCT], densities[OUR_PRODUCT],
        shared_norm, axis_max, "a",
    )
    rows.append(
        _asymmetric_metric_row(
            forest, OUR_PRODUCT, frames[OUR_PRODUCT]
        )
    )
    our_axis.set_xlabel("GEDI RH95 TEST (m)")
    our_axis.set_ylabel("Canopy-height estimate (m)")

    for letter, product in zip("bcde", ASYMMETRIC_PRODUCT_ORDER):
        axis = comparison_axes[product]
        if product not in cfg["products"] or product not in frames:
            _draw_asymmetric_unavailable(axis, product, axis_max, letter)
            rows.append({
                "forest": forest,
                "product": product,
                "label": ASYMMETRIC_LABELS[product],
                "status": "NOT_EVALUABLE",
                "n": 0,
                "r2": np.nan,
                "corr": np.nan,
                "rmse_m": np.nan,
                "mae_m": np.nan,
                "bias_m": np.nan,
            })
        else:
            mappable = _draw_asymmetric_scatter(
                axis, forest, product,
                frames[product], densities[product],
                shared_norm, axis_max, letter,
            )
            rows.append(
                _asymmetric_metric_row(
                    forest, product, frames[product]
                )
            )

    # One common Y scale: labels are retained only on Our Model.
    for product, axis in comparison_axes.items():
        axis.tick_params(axis="y", labelleft=False)
    # X labels only along the bottom of the compact 2 × 2 comparison grid.
    comparison_axes["Lang 2020"].tick_params(axis="x", labelbottom=False)
    comparison_axes["Pauls 2020"].tick_params(axis="x", labelbottom=False)
    comparison_axes["Meta/Tolan 2023"].set_xlabel("GEDI RH95 TEST (m)")
    comparison_axes["GFCH 2019"].set_xlabel("GEDI RH95 TEST (m)")

    # Keep only one spine at each internal comparison-grid boundary.
    comparison_axes["Lang 2020"].spines["right"].set_visible(False)
    comparison_axes["Meta/Tolan 2023"].spines["right"].set_visible(False)
    comparison_axes["Lang 2020"].spines["bottom"].set_visible(False)
    comparison_axes["Pauls 2020"].spines["bottom"].set_visible(False)

    figure.canvas.draw()
    top_left = comparison_axes["Lang 2020"].get_position()
    top_right = comparison_axes["Pauls 2020"].get_position()
    bottom_left = comparison_axes["Meta/Tolan 2023"].get_position()
    bottom_right = comparison_axes["GFCH 2019"].get_position()
    gap_tolerance = 3e-3
    horizontal_gaps = (
        abs(top_left.x1 - top_right.x0),
        abs(bottom_left.x1 - bottom_right.x0),
    )
    vertical_gaps = (
        abs(top_left.y0 - bottom_left.y1),
        abs(top_right.y0 - bottom_right.y1),
    )
    if max(horizontal_gaps + vertical_gaps) > gap_tolerance:
        raise AssertionError(
            "Comparison-grid zero-gap guard failed: "
            f"horizontal={horizontal_gaps}, vertical={vertical_gaps}"
        )

    # Compact, centred density bar outside the comparison grid.
    grid_top = top_right.y1
    grid_bottom = bottom_right.y0
    bar_height = 0.52 * (grid_top - grid_bottom)
    bar_bottom = 0.5 * (grid_top + grid_bottom - bar_height)
    colorbar_axis = figure.add_axes([
        top_right.x1 + 0.018,
        bar_bottom,
        0.012,
        bar_height,
    ])
    scalar_mappable = plt.cm.ScalarMappable(
        norm=shared_norm, cmap="viridis"
    )
    scalar_mappable.set_array([])
    colorbar = figure.colorbar(
        scalar_mappable, cax=colorbar_axis
    )
    colorbar.set_ticks(colorbar_ticks)
    colorbar.set_ticklabels([
        f"{int(value):,}" for value in colorbar_ticks
    ])
    colorbar.ax.minorticks_off()
    colorbar.ax.tick_params(labelsize=8)
    colorbar.set_label(
        "Number of samples",
        fontsize=8.3,
    )

    stem = f"{forest}_article_asymmetric_scatter_zero_gap"
    export_article_figure(figure, stem)
    metrics = pd.DataFrame(rows).assign(
        evaluation_min_m=2.0,
        evaluation_max_m=axis_max,
        support_policy="maximum valid canonical TEST support per product",
    )
    metrics_path = OUT_DIR / f"{stem}_metrics.csv"
    metrics.to_csv(metrics_path, index=False)
    print("Saved:", metrics_path)
    plt.show()
    plt.close(figure)
    return metrics


asymmetric_article_metrics_v3 = []
for forest in SITES:
    print(f"\n[ASYMMETRIC ZERO-GAP SCATTER] {forest}")
    asymmetric_article_metrics_v3.append(
        plot_asymmetric_article_scatter_v3(forest)
    )

asymmetric_article_metrics_v3 = pd.concat(
    asymmetric_article_metrics_v3,
    ignore_index=True,
)
display(asymmetric_article_metrics_v3)
print("[PASS] Our Model isolated on the left.")
print("[PASS] Four comparison CHMs arranged in a zero-gap 2 × 2 grid.")
print("[PASS] Scatter and height-distribution figures are independent.")


## Quantitative gain of Our Model over global CHMs

Two complementary tables are generated to avoid an unfair comparison caused by
different raster coverages.

### A. ECHOSAT-style absolute performance table

Within each forest, every method is evaluated on the **strict intersection of
valid canonical GEDI TEST footprints across all evaluable products**. Therefore,
all rows in a forest have exactly the same observations and the same `n`.

The table reports MAE, MSE, RMSE, MAPE, R², Pearson correlation and bias. Lower
MAE/MSE/RMSE/MAPE is better; higher R²/correlation is better.

### B. Direct paired gain table

Our Model is also compared independently with each global CHM using the maximum
pairwise intersection of valid TEST footprints. This maximizes statistical power
while keeping the two methods of each comparison on exactly the same shots.

For MAE and RMSE:

\[
\mathrm{Gain}_{\%}
=100\frac{E_{\mathrm{baseline}}-E_{\mathrm{ours}}}
{E_{\mathrm{baseline}}}.
\]

A positive value means that Our Model reduces the error. For R² and correlation,
the gain is the direct difference `Our Model − baseline`. TEST is used only for
final reporting; no model or hyperparameter is selected from these tables.


<!-- STEP_ARTICLE_COMPACT_LATEX_READY_V1 -->
## Article-ready figure and comparison protocol

The exported PNG, SVG and PDF panels intentionally contain only information
that must remain attached to the axes: product names, ecosystem row labels,
metric boxes, units and panel letters. The complete scientific caption,
protocol description and figure title must be written in LaTeX. This avoids
duplicating text and wasting page area.

The color scale is labelled **Number of samples**. It represents the number of
paired GEDI–CHM observations inside each 1 × 1 m observed-versus-predicted bin.
It must not be called *Number of pixels*, because GEDI shots are not unique map
pixels.

### Why two performance tables are reported

1. **Strict common performance table.** All available products within one
   forest are evaluated on the intersection of shots valid for every product.
   Consequently, every method has exactly the same `n`. This is the fairest
   single-table ranking, but a product with incomplete coverage—especially
   Potapov/GFCH—can strongly reduce the common sample.
2. **Pairwise gain table.** Our Model is compared separately with each external
   CHM on the exact intersection available for that pair. The sample size
   therefore changes between baselines, but substantially more TEST shots can
   be retained. This table directly quantifies the improvement over each
   competitor.

For the main manuscript, use the strict-common table as the primary benchmark.
Use the pairwise gain table as a complementary robustness analysis or in the
supplementary material. Never compare values from rows having different `n`
without explicitly stating the pairwise-support protocol.


In [ ]:
# STEP_ARTICLE_OUR_MODEL_GAIN_TABLES_V1
GAIN_PRODUCT_ORDER = [
    OUR_PRODUCT,
    "Lang 2020",
    "Pauls 2020",
    "Meta/Tolan 2023",
    "GFCH 2019",
]

GAIN_LABELS = {
    OUR_PRODUCT: "Our Model",
    "Lang 2020": "Lang et al. (L23)",
    "Pauls 2020": "Pauls et al. (Pa24)",
    "Meta/Tolan 2023": "Tolan et al. (T24)",
    "GFCH 2019": "Potapov et al. (P21)",
}


def article_table_metrics(frame):
    observed = frame["rh95"].to_numpy(float)
    predicted = frame["prediction"].to_numpy(float)
    finite = (
        np.isfinite(observed)
        & np.isfinite(predicted)
        & (observed > 0.0)
    )
    observed = observed[finite]
    predicted = predicted[finite]
    if observed.size < 2:
        raise RuntimeError(f"Insufficient paired observations: n={observed.size}")

    residual = predicted - observed
    mse = float(np.mean(residual ** 2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(residual)))
    mape = float(100.0 * np.mean(np.abs(residual / observed)))
    bias = float(np.mean(residual))
    denominator = float(np.sum((observed - observed.mean()) ** 2))
    r2 = (
        float(1.0 - np.sum(residual ** 2) / denominator)
        if denominator > 0.0 else np.nan
    )
    corr = (
        float(np.corrcoef(observed, predicted)[0, 1])
        if np.std(observed) > 0.0 and np.std(predicted) > 0.0
        else np.nan
    )
    slope = (
        float(np.polyfit(observed, predicted, 1)[0])
        if np.std(observed) > 0.0 else np.nan
    )
    return {
        "n": int(observed.size),
        "MAE_m": mae,
        "MSE_m2": mse,
        "RMSE_m": rmse,
        "MAPE_pct": mape,
        "R2": r2,
        "Corr": corr,
        "Bias_m": bias,
        "Slope": slope,
    }


def _unique_product_frame(forest, product):
    frame = valid_samples.loc[
        valid_samples["forest"].eq(forest)
        & valid_samples["product"].eq(product)
    ].copy()
    frame["shot_id"] = frame["shot_id"].astype(str)
    if frame["shot_id"].duplicated().any():
        duplicates = int(frame["shot_id"].duplicated().sum())
        raise AssertionError(
            f"{forest}/{product}: {duplicates} duplicated shot_id values"
        )
    return frame


def strict_common_performance_table(forest):
    cfg = SITES[forest]
    products = [
        product for product in GAIN_PRODUCT_ORDER
        if product in cfg["products"]
    ]
    frames = {
        product: _unique_product_frame(forest, product)
        for product in products
    }
    id_sets = {
        product: set(frame["shot_id"])
        for product, frame in frames.items()
    }
    common_ids = set.intersection(*id_sets.values())
    if len(common_ids) < 30:
        raise RuntimeError(
            f"{forest}: strict all-product TEST support too small "
            f"(n={len(common_ids)})"
        )

    rows = []
    expected_ids = frozenset(common_ids)
    for product in products:
        frame = frames[product].loc[
            frames[product]["shot_id"].isin(common_ids)
        ].copy()
        if frozenset(frame["shot_id"]) != expected_ids:
            raise AssertionError(
                f"{forest}/{product}: strict common shot_id mismatch"
            )
        metric = article_table_metrics(frame)
        if metric["n"] != len(common_ids):
            raise AssertionError(
                f"{forest}/{product}: metric n={metric['n']} "
                f"!= common n={len(common_ids)}"
            )
        rows.append({
            "Forest": forest,
            "Method": GAIN_LABELS[product],
            "Product_key": product,
            **metric,
        })

    table = pd.DataFrame(rows)
    if table["n"].nunique() != 1:
        raise AssertionError(f"{forest}: method rows do not share the same n")

    external = table.loc[~table["Product_key"].eq(OUR_PRODUCT)].copy()
    ours = table.loc[table["Product_key"].eq(OUR_PRODUCT)].iloc[0]
    best_external = {
        "MAE_m": float(external["MAE_m"].min()),
        "MSE_m2": float(external["MSE_m2"].min()),
        "RMSE_m": float(external["RMSE_m"].min()),
        "MAPE_pct": float(external["MAPE_pct"].min()),
        "R2": float(external["R2"].max()),
        "Corr": float(external["Corr"].max()),
    }
    gain = {
        "Forest": forest,
        "strict_common_n": int(len(common_ids)),
        "best_external_MAE_m": best_external["MAE_m"],
        "ours_MAE_m": float(ours["MAE_m"]),
        "MAE_reduction_m": best_external["MAE_m"] - float(ours["MAE_m"]),
        "MAE_reduction_pct": (
            100.0 * (best_external["MAE_m"] - float(ours["MAE_m"]))
            / best_external["MAE_m"]
        ),
        "best_external_RMSE_m": best_external["RMSE_m"],
        "ours_RMSE_m": float(ours["RMSE_m"]),
        "RMSE_reduction_m": best_external["RMSE_m"] - float(ours["RMSE_m"]),
        "RMSE_reduction_pct": (
            100.0 * (best_external["RMSE_m"] - float(ours["RMSE_m"]))
            / best_external["RMSE_m"]
        ),
        "best_external_R2": best_external["R2"],
        "ours_R2": float(ours["R2"]),
        "R2_gain": float(ours["R2"]) - best_external["R2"],
        "best_external_Corr": best_external["Corr"],
        "ours_Corr": float(ours["Corr"]),
        "Corr_gain": float(ours["Corr"]) - best_external["Corr"],
    }
    # Publication order: the weakest external baseline is placed at
    # the top, progressively better external methods follow, and Our Model is
    # anchored as the final bold reference row, as in the ECHOSAT table.
    table["_ours_last"] = table["Product_key"].eq(OUR_PRODUCT).astype(int)
    table = (
        table.sort_values(
            ["_ours_last", "MAE_m"],
            ascending=[True, False],
            kind="mergesort",
        )
        .drop(columns=["_ours_last"])
        .reset_index(drop=True)
    )
    if table.iloc[-1]["Product_key"] != OUR_PRODUCT:
        raise AssertionError(f"{forest}: Our Model must be the final table row")
    return table, pd.DataFrame([gain])


def pairwise_gain_table(forest):
    cfg = SITES[forest]
    ours_all = _unique_product_frame(forest, OUR_PRODUCT)
    ours_ids = set(ours_all["shot_id"])
    rows = []

    for competitor in GAIN_PRODUCT_ORDER[1:]:
        if competitor not in cfg["products"]:
            rows.append({
                "Forest": forest,
                "Baseline": GAIN_LABELS[competitor],
                "status": "NOT_EVALUABLE",
                "paired_n": 0,
            })
            continue

        baseline_all = _unique_product_frame(forest, competitor)
        pair_ids = ours_ids.intersection(set(baseline_all["shot_id"]))
        if len(pair_ids) < 30:
            rows.append({
                "Forest": forest,
                "Baseline": GAIN_LABELS[competitor],
                "status": "INSUFFICIENT_COMMON_SUPPORT",
                "paired_n": int(len(pair_ids)),
            })
            continue

        ours = ours_all.loc[ours_all["shot_id"].isin(pair_ids)].copy()
        baseline = baseline_all.loc[
            baseline_all["shot_id"].isin(pair_ids)
        ].copy()
        if frozenset(ours["shot_id"]) != frozenset(baseline["shot_id"]):
            raise AssertionError(
                f"{forest}/{competitor}: pairwise shot_id mismatch"
            )

        our_metric = article_table_metrics(ours)
        baseline_metric = article_table_metrics(baseline)
        if our_metric["n"] != baseline_metric["n"]:
            raise AssertionError(
                f"{forest}/{competitor}: paired n mismatch"
            )

        mae_gain = baseline_metric["MAE_m"] - our_metric["MAE_m"]
        rmse_gain = baseline_metric["RMSE_m"] - our_metric["RMSE_m"]
        rows.append({
            "Forest": forest,
            "Baseline": GAIN_LABELS[competitor],
            "status": "PASS",
            "paired_n": int(our_metric["n"]),
            "Our_MAE_m": our_metric["MAE_m"],
            "Baseline_MAE_m": baseline_metric["MAE_m"],
            "MAE_reduction_m": mae_gain,
            "MAE_reduction_pct": (
                100.0 * mae_gain / baseline_metric["MAE_m"]
            ),
            "Our_RMSE_m": our_metric["RMSE_m"],
            "Baseline_RMSE_m": baseline_metric["RMSE_m"],
            "RMSE_reduction_m": rmse_gain,
            "RMSE_reduction_pct": (
                100.0 * rmse_gain / baseline_metric["RMSE_m"]
            ),
            "Our_R2": our_metric["R2"],
            "Baseline_R2": baseline_metric["R2"],
            "R2_gain": our_metric["R2"] - baseline_metric["R2"],
            "Our_Corr": our_metric["Corr"],
            "Baseline_Corr": baseline_metric["Corr"],
            "Corr_gain": our_metric["Corr"] - baseline_metric["Corr"],
        })
    return pd.DataFrame(rows)


def _strict_publication_frame(table):
    """Compact ECHOSAT-style columns used in the manuscript."""
    visible = table.loc[:, [
        "Method", "n", "MAE_m", "MSE_m2", "RMSE_m",
        "MAPE_pct", "R2", "Corr",
    ]].copy()
    return visible.rename(columns={
        "n": "n",
        "MAE_m": "MAE (m) ↓",
        "MSE_m2": "MSE (m²) ↓",
        "RMSE_m": "RMSE (m) ↓",
        "MAPE_pct": "MAPE (%) ↓",
        "R2": "R² ↑",
        "Corr": "r ↑",
    })


def _performance_styler(table):
    visible = _strict_publication_frame(table)
    numeric_formats = {
        "n": "{:,.0f}",
        "MAE (m) ↓": "{:.2f}",
        "MSE (m²) ↓": "{:.2f}",
        "RMSE (m) ↓": "{:.2f}",
        "MAPE (%) ↓": "{:.1f}",
        "R² ↑": "{:.2f}",
        "r ↑": "{:.2f}",
    }

    def highlight_ours(row):
        if row["Method"] == "Our Model":
            return ["font-weight: bold" for _ in row]
        return ["" for _ in row]

    return (
        visible.style
        .format(numeric_formats)
        .apply(highlight_ours, axis=1)
        .hide(axis="index")
    )


def _write_latex_table(table, destination, caption, label):
    """Generic supplementary-table writer."""
    latex_table = table.to_latex(
        index=False,
        float_format=lambda value: f"{value:.3f}",
        caption=caption,
        label=label,
        escape=True,
    )
    destination.write_text(latex_table, encoding="utf-8")
    print("Saved:", destination)


def _write_strict_publication_latex(table, destination, caption, label):
    """Native booktabs table with arrows and a bold final Our Model row."""
    rows = []
    for _, row in table.iterrows():
        values = [
            str(row["Method"]),
            f'{int(row["n"]):,}',
            f'{row["MAE_m"]:.2f}',
            f'{row["MSE_m2"]:.2f}',
            f'{row["RMSE_m"]:.2f}',
            f'{row["MAPE_pct"]:.1f}',
            f'{row["R2"]:.2f}',
            f'{row["Corr"]:.2f}',
        ]
        if row["Product_key"] == OUR_PRODUCT:
            values = [rf"\textbf{{{value}}}" for value in values]
        rows.append(" & ".join(values) + r" \\")

    body = "\n".join(rows)
    latex = rf"""\begin{{table}}[t]
\centering
\caption{{{caption}}}
\label{{{label}}}
\small
\setlength{{\tabcolsep}}{{4.5pt}}
\begin{{tabular}}{{lrrrrrrr}}
\toprule
Method & $n$ & MAE (m)$\downarrow$ & MSE (m$^2$)$\downarrow$ &
RMSE (m)$\downarrow$ & MAPE (\%)$\downarrow$ & $R^2\uparrow$ & $r\uparrow$ \\
\midrule
{body}
\bottomrule
\end{{tabular}}
\end{{table}}
"""
    destination.write_text(latex, encoding="utf-8")
    print("Saved:", destination)


strict_tables = []
best_external_gain_tables = []
pairwise_gain_tables = []

for forest in SITES:
    print("\n" + "=" * 90)
    print(f"{forest} — STRICT COMMON PERFORMANCE AND PAIRED GAINS")
    print("=" * 90)
    strict_table, best_gain = strict_common_performance_table(forest)
    pairwise_table = pairwise_gain_table(forest)
    strict_tables.append(strict_table)
    best_external_gain_tables.append(best_gain)
    pairwise_gain_tables.append(pairwise_table)

    print("Publication order (top to bottom):", " > ".join(strict_table["Method"]))
    display(_performance_styler(strict_table))
    display(pairwise_table.style.format(precision=3).hide(axis="index"))

    strict_csv = OUT_DIR / f"{forest}_ECHOSAT_style_strict_common_metrics.csv"
    strict_tex = OUT_DIR / f"{forest}_ECHOSAT_style_strict_common_metrics.tex"
    pair_csv = OUT_DIR / f"{forest}_OurModel_pairwise_gains.csv"
    pair_tex = OUT_DIR / f"{forest}_OurModel_pairwise_gains.tex"
    strict_table.to_csv(strict_csv, index=False)
    pairwise_table.to_csv(pair_csv, index=False)
    print("Saved:", strict_csv)
    print("Saved:", pair_csv)
    _write_strict_publication_latex(
        strict_table,
        strict_tex,
        (
            f"Canopy-height performance in {forest} on the strict common "
            "canonical GEDI TEST support."
        ),
        f"tab:{forest.lower()}_chm_strict_common",
    )
    _write_latex_table(
        pairwise_table,
        pair_tex,
        (
            f"Paired performance gains of Our Model over global CHMs "
            f"in {forest}."
        ),
        f"tab:{forest.lower()}_chm_pairwise_gain",
    )

all_strict_performance = pd.concat(strict_tables, ignore_index=True)
all_best_external_gains = pd.concat(
    best_external_gain_tables, ignore_index=True
)
all_pairwise_gains = pd.concat(pairwise_gain_tables, ignore_index=True)

all_strict_performance.to_csv(
    OUT_DIR / "ALL_FORESTS_ECHOSAT_style_strict_common_metrics.csv",
    index=False,
)
all_best_external_gains.to_csv(
    OUT_DIR / "ALL_FORESTS_OurModel_gain_vs_best_external.csv",
    index=False,
)
all_pairwise_gains.to_csv(
    OUT_DIR / "ALL_FORESTS_OurModel_pairwise_gains.csv",
    index=False,
)
_write_latex_table(
    all_best_external_gains,
    OUT_DIR / "ALL_FORESTS_OurModel_gain_vs_best_external.tex",
    "Gain of Our Model relative to the best external CHM in each forest.",
    "tab:all_forests_gain_best_external",
)

print("\nGAIN OF OUR MODEL RELATIVE TO THE BEST EXTERNAL CHM")
display(
    all_best_external_gains.style
    .format(precision=3)
    .hide(axis="index")
)
print("[PASS] Absolute tables use identical strict-common TEST shots.")
print("[PASS] Pairwise gains use exact shot_id intersections.")
print("[PASS] Positive MAE/RMSE reductions and R²/Corr gains favour Our Model.")


## Article scatter alternatives — matched GEDI TEST support

The previously generated **maximum-valid-support** matrix is intentionally preserved as a coverage diagnostic. It is not overwritten by the cells below.

Two additional, complementary protocols are exported with distinct filenames:

1. **Pairwise matched support:** Our Model and one global CHM are restricted to the exact same `shot_id` intersection. This is the visual counterpart of the pairwise gain tables and is the recommended protocol for claims of improvement.
2. **Strict common support:** all evaluable products for one forest are restricted to one common `shot_id` intersection. This is the most direct multi-model visual comparison, at the cost of a smaller sample size.

All exports use new, versioned stems. Re-running this cell creates a new version instead of overwriting either the original coverage figure or an earlier matched-support figure.


In [ ]:
# STEP_ARTICLE_MATCHED_SUPPORT_SCATTER_PROPOSALS_V1
from matplotlib.colors import Normalize


MATCHED_EXPORT_EXTENSIONS = ("png", "svg", "pdf")


def _unused_export_stem(base_stem):
    """Return a stem that cannot overwrite an existing article figure."""
    candidate = str(base_stem)
    version = 1
    while any((OUT_DIR / f"{candidate}.{ext}").exists() for ext in MATCHED_EXPORT_EXTENSIONS):
        version += 1
        candidate = f"{base_stem}_run{version:02d}"
    return candidate


def export_matched_figure(figure, base_stem):
    stem = _unused_export_stem(base_stem)
    destinations = []
    for extension in MATCHED_EXPORT_EXTENSIONS:
        destination = OUT_DIR / f"{stem}.{extension}"
        if destination.exists():
            raise FileExistsError(destination)
        figure.savefig(
            destination,
            dpi=600 if extension == "png" else None,
            bbox_inches="tight",
            facecolor="white",
        )
        destinations.append(destination)
        print("Saved:", destination)
    return stem, destinations


def _matched_product_frames(forest, products, minimum_n=30):
    frames = {product: _unique_product_frame(forest, product) for product in products}
    common_ids = set.intersection(*(set(frame["shot_id"]) for frame in frames.values()))
    if len(common_ids) < minimum_n:
        return {}, common_ids
    matched = {
        product: frame.loc[frame["shot_id"].isin(common_ids)].copy()
        for product, frame in frames.items()
    }
    expected = frozenset(common_ids)
    for product, frame in matched.items():
        if frozenset(frame["shot_id"]) != expected:
            raise AssertionError(f"{forest}/{product}: matched shot_id mismatch")
    return matched, common_ids


def _scatter_density(frame, axis_max):
    density = point_concentration_1m(frame, axis_max)
    order = np.argsort(density, kind="mergesort")
    return density, order


def _draw_matched_scatter(axis, frame, axis_max, title, norm, density):
    metric = article_table_metrics(frame)
    order = np.argsort(density, kind="mergesort")
    points = axis.scatter(
        frame["rh95"].to_numpy(float)[order],
        frame["prediction"].to_numpy(float)[order],
        c=density[order],
        cmap="viridis",
        norm=norm,
        s=8.0,
        alpha=0.82,
        linewidths=0,
        rasterized=True,
    )
    axis.plot([0, axis_max], [0, axis_max], color="black", linestyle="--", linewidth=0.9)
    axis.set_xlim(0, axis_max)
    axis.set_ylim(0, axis_max)
    axis.set_aspect("equal", adjustable="box")
    axis.set_title(title, fontsize=10.5, fontweight="bold")
    axis.grid(True, alpha=0.20, linewidth=0.6)
    axis.text(
        0.025,
        0.975,
        (
            f"R²={metric['R2']:.2f}\n"
            f"RMSE={metric['RMSE_m']:.2f} m\n"
            f"MAE={metric['MAE_m']:.2f} m\n"
            f"Bias={metric['Bias_m']:+.2f} m\n"
            f"Corr={metric['Corr']:.2f}\n"
            f"n={metric['n']:,}"
        ),
        transform=axis.transAxes,
        ha="left",
        va="top",
        fontsize=8.5,
        bbox={"facecolor": "white", "alpha": 0.76, "edgecolor": "0.75", "pad": 2.0},
    )
    return points, metric


def plot_pairwise_matched_scatter_proposal(forest):
    """One row per baseline; both panels in a row use identical GEDI shots."""
    cfg = SITES[forest]
    axis_max = float(cfg["eval_max"])
    ours_all = _unique_product_frame(forest, OUR_PRODUCT)
    rows = []
    pairs = []

    for competitor in GAIN_PRODUCT_ORDER[1:]:
        if competitor not in cfg["products"]:
            rows.append({
                "forest": forest,
                "protocol": "pairwise_exact_shot_id_intersection",
                "baseline": GAIN_LABELS[competitor],
                "status": "NOT_EVALUABLE",
                "paired_n": 0,
            })
            continue
        competitor_all = _unique_product_frame(forest, competitor)
        pair_ids = set(ours_all["shot_id"]).intersection(set(competitor_all["shot_id"]))
        if len(pair_ids) < 30:
            rows.append({
                "forest": forest,
                "protocol": "pairwise_exact_shot_id_intersection",
                "baseline": GAIN_LABELS[competitor],
                "status": "INSUFFICIENT_COMMON_SUPPORT",
                "paired_n": int(len(pair_ids)),
            })
            continue
        ours = ours_all.loc[ours_all["shot_id"].isin(pair_ids)].copy()
        baseline = competitor_all.loc[competitor_all["shot_id"].isin(pair_ids)].copy()
        if frozenset(ours["shot_id"]) != frozenset(baseline["shot_id"]):
            raise AssertionError(f"{forest}/{competitor}: pairwise shot_id mismatch")
        pairs.append((competitor, ours, baseline))

    if not pairs:
        raise RuntimeError(f"{forest}: no evaluable pairwise comparison")

    figure = plt.figure(figsize=(10.4, 3.65 * len(pairs)))
    grid = figure.add_gridspec(
        len(pairs), 3,
        width_ratios=[1.0, 1.0, 0.035],
        wspace=0.08,
        hspace=0.22,
    )

    for row_index, (competitor, ours, baseline) in enumerate(pairs):
        density_ours, _ = _scatter_density(ours, axis_max)
        density_baseline, _ = _scatter_density(baseline, axis_max)
        maximum_density = max(float(np.max(density_ours)), float(np.max(density_baseline)))
        upper, ticks = nice_colorbar_scale(maximum_density)
        norm = Normalize(vmin=0.0, vmax=upper)
        left_axis = figure.add_subplot(grid[row_index, 0])
        right_axis = figure.add_subplot(grid[row_index, 1], sharex=left_axis, sharey=left_axis)
        colorbar_axis = figure.add_subplot(grid[row_index, 2])

        points, our_metric = _draw_matched_scatter(
            left_axis, ours, axis_max, "Our Model", norm, density_ours
        )
        _, baseline_metric = _draw_matched_scatter(
            right_axis, baseline, axis_max, GAIN_LABELS[competitor], norm, density_baseline
        )
        if our_metric["n"] != baseline_metric["n"]:
            raise AssertionError(f"{forest}/{competitor}: paired n mismatch")

        left_axis.set_ylabel("Predicted height (m)")
        if row_index == len(pairs) - 1:
            left_axis.set_xlabel("RH95 from GEDI waveforms (m)")
            right_axis.set_xlabel("RH95 from GEDI waveforms (m)")
        else:
            left_axis.tick_params(labelbottom=False)
            right_axis.tick_params(labelbottom=False)
        right_axis.tick_params(labelleft=False)

        colorbar = figure.colorbar(points, cax=colorbar_axis)
        colorbar.set_ticks(ticks)
        colorbar.set_ticklabels([f"{int(value):,}" for value in ticks])
        colorbar.ax.minorticks_off()
        colorbar.set_label("Number of samples", fontsize=8.5)

        rows.append({
            "forest": forest,
            "protocol": "pairwise_exact_shot_id_intersection",
            "baseline": GAIN_LABELS[competitor],
            "status": "PASS",
            "paired_n": int(our_metric["n"]),
            "Our_MAE_m": our_metric["MAE_m"],
            "Baseline_MAE_m": baseline_metric["MAE_m"],
            "Our_RMSE_m": our_metric["RMSE_m"],
            "Baseline_RMSE_m": baseline_metric["RMSE_m"],
            "Our_R2": our_metric["R2"],
            "Baseline_R2": baseline_metric["R2"],
        })

    figure.align_ylabels()
    stem, _ = export_matched_figure(
        figure,
        f"{forest}_PAIRWISE_MATCHED_OurModel_vs_global_CHMs_v1",
    )
    plt.show()
    plt.close(figure)
    table = pd.DataFrame(rows)
    table_path = OUT_DIR / f"{stem}_metrics.csv"
    table.to_csv(table_path, index=False)
    print("Saved:", table_path)
    display(table)
    return table


def plot_strict_common_scatter_proposal(forest):
    """All evaluable products use one strict common GEDI shot intersection."""
    cfg = SITES[forest]
    axis_max = float(cfg["eval_max"])
    products = [product for product in GAIN_PRODUCT_ORDER if product in cfg["products"]]
    matched, common_ids = _matched_product_frames(forest, products, minimum_n=30)
    if not matched:
        raise RuntimeError(f"{forest}: strict common support is insufficient (n={len(common_ids)})")

    densities = {
        product: point_concentration_1m(matched[product], axis_max)
        for product in products
    }
    maximum_density = max(float(np.max(values)) for values in densities.values())
    upper, ticks = nice_colorbar_scale(maximum_density)
    norm = Normalize(vmin=0.0, vmax=upper)

    figure = plt.figure(figsize=(3.25 * len(products) + 0.55, 3.45))
    grid = figure.add_gridspec(
        1, len(products) + 1,
        width_ratios=[1.0] * len(products) + [0.035],
        wspace=0.04,
    )
    metric_rows = []
    mappable = None
    for column_index, product in enumerate(products):
        axis = figure.add_subplot(grid[0, column_index])
        mappable, metric = _draw_matched_scatter(
            axis,
            matched[product],
            axis_max,
            GAIN_LABELS[product],
            norm,
            densities[product],
        )
        axis.set_xlabel("RH95 from GEDI waveforms (m)")
        if column_index == 0:
            axis.set_ylabel("Predicted height (m)")
        else:
            axis.tick_params(labelleft=False)
        metric_rows.append({
            "forest": forest,
            "protocol": "strict_all_product_shot_id_intersection",
            "product": GAIN_LABELS[product],
            **metric,
        })

    colorbar_axis = figure.add_subplot(grid[0, -1])
    colorbar = figure.colorbar(mappable, cax=colorbar_axis)
    colorbar.set_ticks(ticks)
    colorbar.set_ticklabels([f"{int(value):,}" for value in ticks])
    colorbar.ax.minorticks_off()
    colorbar.set_label("Number of samples", fontsize=8.5)

    stem, _ = export_matched_figure(
        figure,
        f"{forest}_STRICT_COMMON_all_CHM_scatter_v1",
    )
    plt.show()
    plt.close(figure)
    table = pd.DataFrame(metric_rows)
    table_path = OUT_DIR / f"{stem}_metrics.csv"
    table.to_csv(table_path, index=False)
    print("Saved:", table_path)
    display(table)
    return table


pairwise_matched_scatter_tables = []
strict_common_scatter_tables = []
for forest_name in ("Ifran", "Maamoura", "Agadir"):
    print("\n" + "=" * 92)
    print(f"{forest_name} — matched-support scatter alternatives")
    print("=" * 92)
    pairwise_matched_scatter_tables.append(
        plot_pairwise_matched_scatter_proposal(forest_name)
    )
    strict_common_scatter_tables.append(
        plot_strict_common_scatter_proposal(forest_name)
    )

all_pairwise_matched_scatter_metrics = pd.concat(
    pairwise_matched_scatter_tables, ignore_index=True
)
all_strict_common_scatter_metrics = pd.concat(
    strict_common_scatter_tables, ignore_index=True
)

print("\n[PASS] Original maximum-support scatter matrix was not modified or overwritten.")
print("[PASS] Pairwise figures use exact Our Model/baseline shot_id intersections.")
print("[PASS] Strict-common figures use one identical shot_id set for every displayed product.")


## Final article matrix — locked 3 × 5 design

This cell produces the compact comparison requested for the article:

- rows: Ifran, Maamoura and Agadir;
- columns: Our Model, Lang et al. (L23), Pauls et al. (Pa24), Tolan et al. (T24) and Potapov et al. (P21);
- identical 0–45 m axes in every panel;
- zero horizontal and vertical spacing;
- one independent linear density scale per forest;
- maximum valid canonical TEST support for each product;
- metrics reported inside each panel.

The output has its own versioned name and does not overwrite previous matrices.


## Final article matrix — strict common GEDI support

Within each forest row, all evaluable CHM products are now compared on exactly the same GEDI shot IDs. Consequently, `n` is identical across the row. A product declared non-evaluable is not allowed to reduce the support of the remaining products.


In [ ]:
# STEP_FINAL_ARTICLE_MATRIX_STRICT_COMMON_N_V2
FINAL_MATRIX_STEM = "ALL_FORESTS_CHM_SCATTER_MATRIX_STRICT_COMMON_N_V2"

final_article_matrix_metrics = plot_combined_ecosystem_scatter_matrix(
    uniform_axis_45m=True,
    figure_stem_override=FINAL_MATRIX_STEM,
    strict_common_support=True,
)

for forest in [name for name, _ in ECOSYSTEM_ROWS]:
    evaluable = final_article_matrix_metrics[
        final_article_matrix_metrics["forest"].eq(forest)
        & final_article_matrix_metrics["status"].eq("PASS")
    ]
    unique_n = sorted(evaluable["n"].dropna().astype(int).unique())
    assert len(unique_n) == 1, (forest, unique_n)
    print(f"[PASS] {forest}: identical strict-common n={unique_n[0]:,}")

print("PNG:", OUT_DIR / f"{FINAL_MATRIX_STEM}.png")
print("SVG:", OUT_DIR / f"{FINAL_MATRIX_STEM}.svg")
print("PDF:", OUT_DIR / f"{FINAL_MATRIX_STEM}.pdf")
print("Metrics:", OUT_DIR / f"{FINAL_MATRIX_STEM}_metrics.csv")


## Sensitivity figure — strict common support without Potapov P21

Potapov/GFCH has substantially lower valid coverage in Ifran and Maamoura and is not evaluable in Agadir. This complementary matrix excludes P21 before computing the strict GEDI-shot intersection. The remaining four methods are therefore compared on the same, substantially larger support within each forest.


In [ ]:
# STEP_FINAL_ARTICLE_MATRIX_WITHOUT_POTAPOV_V1
MATRIX_WITHOUT_POTAPOV = [
    "Our B4 Phase 2",
    "Lang 2020",
    "Pauls 2020",
    "Meta/Tolan 2023",
]
WITHOUT_POTAPOV_STEM = (
    "ALL_FORESTS_CHM_SCATTER_MATRIX_STRICT_COMMON_N_WITHOUT_P21_V1"
)

_matrix_products_before_without_p21 = list(MATRIX_PRODUCTS)
try:
    MATRIX_PRODUCTS = list(MATRIX_WITHOUT_POTAPOV)
    matrix_without_potapov_metrics = (
        plot_combined_ecosystem_scatter_matrix(
            uniform_axis_45m=True,
            figure_stem_override=WITHOUT_POTAPOV_STEM,
            strict_common_support=True,
        )
    )
finally:
    MATRIX_PRODUCTS = _matrix_products_before_without_p21

for forest in [name for name, _ in ECOSYSTEM_ROWS]:
    forest_rows = matrix_without_potapov_metrics[
        matrix_without_potapov_metrics["forest"].eq(forest)
        & matrix_without_potapov_metrics["status"].eq("PASS")
    ]
    assert set(forest_rows["product"]) == set(MATRIX_WITHOUT_POTAPOV)
    unique_n = sorted(forest_rows["n"].dropna().astype(int).unique())
    assert len(unique_n) == 1, (forest, unique_n)
    print(
        f"[PASS WITHOUT P21] {forest}: "
        f"same n={unique_n[0]:,} for all four methods"
    )

print("PNG:", OUT_DIR / f"{WITHOUT_POTAPOV_STEM}.png")
print("SVG:", OUT_DIR / f"{WITHOUT_POTAPOV_STEM}.svg")
print("PDF:", OUT_DIR / f"{WITHOUT_POTAPOV_STEM}.pdf")
print("Metrics:", OUT_DIR / f"{WITHOUT_POTAPOV_STEM}_metrics.csv")


## LaTeX-clean strict-common scatter matrix

This export omits embedded ecosystem, product and axis prose. Those elements are composed in LaTeX, while numerical ticks, metric boxes and the concentration scales remain in the vector figure.


In [ ]:
# STEP_LATEX_CLEAN_STRICT_COMMON_MATRIX_WITHOUT_P21_V1
# Publication order by accuracy: Our Model, Pauls, Lang, Tolan.
# Text headings and common axis captions are deliberately composed in LaTeX.
LATEX_CLEAN_PRODUCTS = [
    "Our B4 Phase 2",
    "Pauls 2020",
    "Lang 2020",
    "Meta/Tolan 2023",
    "GFCH 2019",
]
LATEX_CLEAN_STEM = "ALL_FORESTS_CHM_SCATTER_MATRIX_STRICT_COMMON_FULL_LATEX_CLEAN"

_matrix_products_before_latex_clean = list(MATRIX_PRODUCTS)
try:
    MATRIX_PRODUCTS = list(LATEX_CLEAN_PRODUCTS)
    latex_clean_matrix_metrics = plot_combined_ecosystem_scatter_matrix(
        uniform_axis_45m=True,
        figure_stem_override=LATEX_CLEAN_STEM,
        strict_common_support=True,
        latex_clean=True,
    )
finally:
    MATRIX_PRODUCTS = _matrix_products_before_latex_clean

for forest in [name for name, _ in ECOSYSTEM_ROWS]:
    rows = latex_clean_matrix_metrics[
        latex_clean_matrix_metrics["forest"].eq(forest)
        & latex_clean_matrix_metrics["status"].eq("PASS")
    ]
    assert set(rows["product"]) == set(LATEX_CLEAN_PRODUCTS)
    assert rows["n"].astype(int).nunique() == 1
    print(f"[PASS LATEX CLEAN] {forest}: common n={int(rows['n'].iloc[0]):,}")


## Final support audit — Agadir

The CHM benchmark uses a spatial footprint support that is distinct from the
temporal T4 evaluator. The raw canonical registry is first sampled against each
map using a 25 m GEDI footprint and an 80% valid-area threshold. The
strict-common figures then retain the exact intersection of `shot_id` values
available for every displayed product. Thus, all panels in an Agadir
strict-common row must use exactly the same 6,401 shots.


In [ ]:
# AGADIR_STRICT_COMMON_SUPPORT_GUARD_V1
_agadir_cfg = SITES["Agadir"]
_agadir_canonical_ids = frozenset(load_test_points("Agadir", _agadir_cfg)["shot_id"].astype(str))
# valid_samples is the canonical product-sample table built above.
_agadir_samples = valid_samples[
    valid_samples["forest"].eq("Agadir")
].copy()
_agadir_products = list(_agadir_cfg["products"])
_agadir_id_sets = {
    product: frozenset(
        _agadir_samples.loc[
            _agadir_samples["product"].eq(product)
            & np.isfinite(_agadir_samples["prediction"]),
            "shot_id",
        ].astype(str)
    )
    for product in _agadir_products
}
_agadir_common_ids = frozenset(set.intersection(*(set(ids) for ids in _agadir_id_sets.values())))

assert _agadir_common_ids <= _agadir_canonical_ids
assert len(_agadir_canonical_ids) == 9206, len(_agadir_canonical_ids)
assert len(_agadir_id_sets[OUR_PRODUCT]) == 6409, len(_agadir_id_sets[OUR_PRODUCT])
assert len(_agadir_common_ids) == 6401, len(_agadir_common_ids)

_agadir_support_table = pd.DataFrame([
    {"support": "Raw canonical TEST registry", "n": len(_agadir_canonical_ids)},
    {"support": "Our annual map valid at ≥80% footprint coverage", "n": len(_agadir_id_sets[OUR_PRODUCT])},
    {"support": "Strict common CHM intersection", "n": len(_agadir_common_ids)},
])
display(_agadir_support_table)
print("[PASS] Agadir strict-common figures use one identical n=6,401 shot_id set.")
